# Module 2 — Feature Engineering
## Phase 1: Load All Cities’ GTFS Data

**Goal:** Load the five core GTFS files for all 8 cities into a single dictionary `data[city][file]`.
Add a `city` column and prefix every ID column with the city name so IDs stay unique across cities.

> **Rule:** No values are cleaned or changed in this phase — loading and labelling only.

In [1]:
# Cell 1: Imports and project configuration
from pathlib import Path
import pandas as pd

BASE_DIR = Path(".")

CITIES = {
    "ahmedabad" : "GTFS AHEMDABAD",
    "bengaluru"  : "GTFS BENGALURU",
    "davangere"  : "GTFS DAVANGARE",
    "delhi"      : "GTFS DELHI",
    "hdbrt"      : "GTFS HDBRT",
    "mumbai"     : "GTFS MUMBAI",
    "pune"       : "GTFS PUNE",
    "rajkot"     : "GTFS RAJKOT",
}

CSV_CITIES  = {"delhi"}
CORE_FILES  = ["routes", "trips", "stop_times", "stops", "calendar"]
OPTIONAL_FILES = ["shapes", "frequencies"]

ID_COLS = {
    "routes"     : ["route_id", "agency_id"],
    "trips"      : ["route_id", "trip_id", "service_id", "shape_id"],
    "stop_times" : ["trip_id", "stop_id"],
    "stops"      : ["stop_id", "zone_id", "parent_station"],
    "calendar"   : ["service_id"],
}

print("Configuration ready.")
print(f"  Cities : {list(CITIES.keys())}")
print(f"  Files  : {CORE_FILES}")

Configuration ready.
  Cities : ['ahmedabad', 'bengaluru', 'davangere', 'delhi', 'hdbrt', 'mumbai', 'pune', 'rajkot']
  Files  : ['routes', 'trips', 'stop_times', 'stops', 'calendar']


### What Cell 1 sets up

- **`CITIES`** maps short city keys to real folder names on disk.
- **`CSV_CITIES`** records that Delhi uses `.csv`; all others use `.txt`.
- **`ID_COLS`** lists identifier columns that must be read as strings so values like `007` are never truncated to `7`.

In [2]:
# Cell 2: Load all core files for every city
data = {}

for city, folder in CITIES.items():
    city_path = BASE_DIR / folder
    ext       = ".csv" if city in CSV_CITIES else ".txt"
    data[city] = {}

    for file_name in CORE_FILES:
        file_path = city_path / f"{file_name}{ext}"
        try:
            header_cols = pd.read_csv(file_path, nrows=0).columns.tolist()
            dtype_map   = {c: str for c in ID_COLS.get(file_name, []) if c in header_cols}
            df = pd.read_csv(file_path, dtype=dtype_map, low_memory=False, keep_default_na=True)
            data[city][file_name] = df
            print(f"  OK  {city:12s} | {file_name:12s} | {df.shape[0]:>9,} rows x {df.shape[1]} cols")
        except FileNotFoundError:
            print(f"  !! MISSING  {city:12s} | {file_name:12s} | {file_path}")
            data[city][file_name] = None

  OK  ahmedabad    | routes       |       978 rows x 5 cols
  OK  ahmedabad    | trips        |    14,323 rows x 6 cols
  OK  ahmedabad    | stop_times   |   646,854 rows x 6 cols
  OK  ahmedabad    | stops        |     3,235 rows x 4 cols
  OK  ahmedabad    | calendar     |         1 rows x 10 cols
  OK  bengaluru    | routes       |     4,434 rows x 5 cols
  OK  bengaluru    | trips        |    57,836 rows x 6 cols
  OK  bengaluru    | stop_times   | 1,540,686 rows x 6 cols
  OK  bengaluru    | stops        |     9,960 rows x 9 cols
  OK  bengaluru    | calendar     |         1 rows x 10 cols
  OK  davangere    | routes       |        93 rows x 5 cols
  OK  davangere    | trips        |       586 rows x 3 cols
  OK  davangere    | stop_times   |    17,783 rows x 5 cols
  OK  davangere    | stops        |       146 rows x 4 cols
  OK  davangere    | calendar     |         1 rows x 10 cols
  OK  delhi        | routes       |     1,270 rows x 5 cols
  OK  delhi        | trips        |  

### Why we peek at the header before loading

Column sets differ between cities. Peeking with `nrows=0` lets us build a `dtype_map` only for columns that actually exist, avoiding errors on cities with fewer columns. `low_memory=False` prevents incorrect type guesses on large files.

In [3]:
# Cell 3: Add 'city' column and prefix every ID with the city name
for city in CITIES:
    for file_name in CORE_FILES:
        df = data[city][file_name]
        if df is None:
            continue
        df.insert(0, "city", city)
        for col in ID_COLS.get(file_name, []):
            if col in df.columns:
                df[col] = city + "_" + df[col].astype(str)
        data[city][file_name] = df

print("City column added and IDs prefixed.")
print("\nSanity check - Pune routes (first 3 rows):")
print(data["pune"]["routes"][["city", "route_id"]].head(3).to_string(index=False))

City column added and IDs prefixed.

Sanity check - Pune routes (first 3 rows):
city route_id
pune   pune_6
pune   pune_7
pune  pune_10


### Why we prefix IDs with the city name

GTFS feeds are made independently; two cities can reuse the same ID values. After prefixing, `route_id = "101"` becomes `"mumbai_101"` and `"pune_101"` — globally unique across the project. The same prefix is applied consistently so joins between files still work.

In [4]:
# Cell 4: Check whether optional files exist
optional_status = {}
for city, folder in CITIES.items():
    city_path = BASE_DIR / folder
    ext = ".csv" if city in CSV_CITIES else ".txt"
    optional_status[city] = {opt: (city_path / f"{opt}{ext}").exists() for opt in OPTIONAL_FILES}
print("Optional file existence check complete.")

Optional file existence check complete.


In [5]:
# Cell 5: Phase 1 summary table
SEP = "=" * 95
print(SEP + "\n  PHASE 1 SUMMARY\n" + SEP)

print("\nA. Row counts per file per city")
hdr = f"{'City':<12} | {'routes':>8} | {'trips':>8} | {'stop_times':>12} | {'stops':>8} | {'calendar':>8}"
print(hdr); print("-" * len(hdr))
for city in CITIES:
    def fmt(f):
        df = data[city][f]
        return f"{len(df):>8,}" if df is not None else f"{'MISSING':>8}"
    print(f"{city:<12} | {fmt('routes')} | {fmt('trips')} | {fmt('stop_times'):>12} | {fmt('stops'):>8} | {fmt('calendar'):>8}")

print("\nB. Columns in 'trips' per city")
for city in CITIES:
    df = data[city]["trips"]
    print(f"  {city:<12}: {list(df.columns) if df is not None else ['MISSING']}")

print("\nC. Columns in 'routes' per city")
for city in CITIES:
    df = data[city]["routes"]
    print(f"  {city:<12}: {list(df.columns) if df is not None else ['MISSING']}")

print("\nD. Optional file availability")
print(f"{'City':<12} | {'shapes':^8} | {'frequencies':^11}"); print("-" * 35)
for city in CITIES:
    s = "YES" if optional_status[city]["shapes"]      else "no"
    f = "YES" if optional_status[city]["frequencies"] else "no"
    print(f"{city:<12} | {s:^8} | {f:^11}")
print("\n" + SEP)

  PHASE 1 SUMMARY

A. Row counts per file per city
City         |   routes |    trips |   stop_times |    stops | calendar
-----------------------------------------------------------------------
ahmedabad    |      978 |   14,323 |      646,854 |    3,235 |        1
bengaluru    |    4,434 |   57,836 |    1,540,686 |    9,960 |        1
davangere    |       93 |      586 |       17,783 |      146 |        1
delhi        |    1,270 |   50,355 |    2,250,290 |    4,192 |        1
hdbrt        |      632 |    8,669 |      194,059 |    2,332 |        2
mumbai       |    1,609 |   99,080 |    2,655,026 |   10,279 |        1
pune         |      628 |   10,128 |      429,302 |    6,745 |        1
rajkot       |      271 |    3,192 |       94,809 |      760 |        1

B. Columns in 'trips' per city
  ahmedabad   : ['city', 'route_id', 'service_id', 'trip_id', 'trip_headsign', 'direction_id', 'shape_id']
  bengaluru   : ['city', 'route_id', 'service_id', 'trip_headsign', 'direction_id', 'shape

### Phase 1 notes

- Large `stop_times` files (Mumbai > 2.65 M rows, Delhi > 2.25 M rows) mean joins will be heavy.
- Columns absent in some cities cannot be used in cross-city features without handling the missing case.
- `shapes.txt` available only for Bengaluru and Pune. `frequencies.txt` absent everywhere.

---
# Phase 2: Data Quality Checks

**Goal:** Inspect the loaded data for problems and **report only** — no values are deleted or changed.

Checks: missing values • duplicate rows and IDs • orphan cross-file IDs • time format • stop coordinates • calendar date ranges.

Results saved to `reports/data_quality_checks.csv`.

In [6]:
# Phase 2, Cell 1: Missing values
missing_records = []
print(f"{'City':<12} {'File':<12} {'Column':<25} {'Missing':>8} {'Pct':>7}")
print("-" * 68)
for city in CITIES:
    for file_name in CORE_FILES:
        df = data[city][file_name]
        if df is None: continue
        mc = df.isnull().sum()
        mc = mc[mc > 0]
        for col, cnt in mc.items():
            pct = round(100.0 * cnt / len(df), 2)
            print(f"{city:<12} {file_name:<12} {col:<25} {cnt:>8,} {pct:>6.2f}%")
            missing_records.append({"city": city, "file": file_name, "column": col,
                                    "missing_count": cnt, "missing_pct": pct})
if not missing_records:
    print("  No missing values found in any city or file.")
missing_df = pd.DataFrame(missing_records)
print(f"\nTotal column-file combinations with missing values: {len(missing_df)}")

City         File         Column                     Missing     Pct
--------------------------------------------------------------------
ahmedabad    trips        shape_id                    14,323 100.00%
bengaluru    stops        parent_station               9,883  99.23%
bengaluru    stops        zone_id                        199   2.00%
bengaluru    stops        stop_desc                      455   4.57%
bengaluru    stops        location_type                9,948  99.88%
bengaluru    stops        platform_code                9,883  99.23%
delhi        routes       route_short_name             1,270 100.00%
delhi        trips        shape_id                    50,355 100.00%
delhi        stops        stop_code                        1   0.02%

Total column-file combinations with missing values: 9


### What missing values mean — and what NOT to do

| Column | Likely meaning | What NOT to do |
|--------|---------------|----------------|
| `stop_lat` / `stop_lon` | No GPS recorded | Do **not** fill with `0` — lat/lon 0 is in the ocean off Africa |
| `shape_id` in trips | No drawn path (normal) | Do **not** fill with a dummy value |
| `arrival_time` / `departure_time` | Agency only records major timepoints | Do **not** interpolate without documenting |
| `direction_id` | Feed omits direction labelling | Do **not** assume 0 = inbound |
| `agency_id` | Single-agency feed (optional) | Do **not** flag as an error |

> All missing values are recorded here. Nothing is changed. Handling decisions are made in a later phase.

In [7]:
# Phase 2, Cell 2: Duplicate rows and duplicate primary IDs
dup_records = []
PRIMARY_ID  = {"routes": "route_id", "trips": "trip_id", "stops": "stop_id"}
print(f"{'City':<12} {'File':<12} {'Check':<25} {'Count':>8}")
print("-" * 62)
for city in CITIES:
    for file_name in CORE_FILES:
        df = data[city][file_name]
        if df is None: continue
        n_dup = int(df.duplicated().sum())
        if n_dup > 0: print(f"{city:<12} {file_name:<12} {'duplicate_rows':<25} {n_dup:>8,}")
        dup_records.append({"city": city, "file": file_name, "check": "duplicate_rows", "count": n_dup})
        if file_name in PRIMARY_ID:
            id_col = PRIMARY_ID[file_name]
            if id_col in df.columns:
                n_id = int(df[id_col].duplicated().sum())
                label = f"duplicate_{id_col}"
                if n_id > 0: print(f"{city:<12} {file_name:<12} {label:<25} {n_id:>8,}")
                dup_records.append({"city": city, "file": file_name, "check": label, "count": n_id})
            else:
                print(f"{city:<12} {file_name:<12} {'[id col missing]':<25} {'N/A':>8}")
dup_df = pd.DataFrame(dup_records)
print(f"\nTotal checks with count > 0: {(dup_df['count'] > 0).sum()}")

City         File         Check                        Count
--------------------------------------------------------------

Total checks with count > 0: 0


### What duplicates mean — and what NOT to do

**Duplicate rows:** File likely concatenated twice. Do **not** drop without verifying all values are truly identical.

**Duplicate `route_id`:** Any join produces a cartesian expansion. Do **not** keep only the first row without checking.

**Duplicate `trip_id`:** Serious — `stop_times` joins on `trip_id`, mixing up schedules. Flag for review.

**Duplicate `stop_id`:** Lat/lon lookups return multiple rows. Do **not** assume both rows are the same stop.

In [8]:
# Phase 2, Cell 3: Cross-file identifier consistency (orphan IDs)
xref_records = []
XREF_CHECKS  = [
    ("trips",      "route_id",   "routes",   "route_id"),
    ("stop_times", "trip_id",    "trips",    "trip_id"),
    ("stop_times", "stop_id",    "stops",    "stop_id"),
    ("trips",      "service_id", "calendar", "service_id"),
]
print(f"{'City':<12} {'Child':<12} {'Col':<12} {'Parent':<12} {'Orphans':>9} {'%':>7}")
print("-" * 68)
for city in CITIES:
    for child_file, child_col, parent_file, parent_col in XREF_CHECKS:
        cdf = data[city][child_file]; pdf = data[city][parent_file]
        if cdf is None or pdf is None: continue
        if child_col not in cdf.columns or parent_col not in pdf.columns:
            xref_records.append({"city": city, "child_file": child_file, "child_col": child_col,
                "parent_file": parent_file, "orphan_count": None, "orphan_pct": None, "note": "col missing"})
            continue
        valid     = set(pdf[parent_col].dropna().unique())
        vals      = cdf[child_col]
        mask      = ~vals.isin(valid) & vals.notna()
        n_orph    = int(mask.sum())
        total     = int(vals.notna().sum())
        pct       = round(100.0 * n_orph / total, 2) if total > 0 else 0.0
        if n_orph > 0:
            print(f"{city:<12} {child_file:<12} {child_col:<12} {parent_file:<12} {n_orph:>9,} {pct:>6.2f}%")
        xref_records.append({"city": city, "child_file": child_file, "child_col": child_col,
            "parent_file": parent_file, "orphan_count": n_orph, "orphan_pct": pct, "note": ""})
xref_df   = pd.DataFrame(xref_records)
n_problems = int((xref_df["orphan_count"].fillna(0) > 0).sum())
print(f"\nCity-check combinations with orphans > 0: {n_problems}")

City         Child        Col          Parent         Orphans       %
--------------------------------------------------------------------

City-check combinations with orphans > 0: 0


### What orphan IDs mean — and what NOT to do

| Orphan | Effect | What NOT to do |
|--------|--------|----------------|
| `trips.route_id` not in `routes` | Trip loses route metadata in any join | Do **not** drop; it may still have valid stop-times |
| `stop_times.trip_id` not in `trips` | Stop-time cannot link to a route | Do **not** assume the trip is inactive |
| `stop_times.stop_id` not in `stops` | Cannot geo-locate the stop event | Do **not** invent a location |
| `trips.service_id` not in `calendar` | Operating days unknown | Do **not** assume it runs every day |

> Below 0.1% is likely noise; above 5% is a structural problem requiring mentor review.

In [9]:
# Phase 2, Cell 4: Time format validation
import re
TIME_PATTERN = re.compile(r'^\d+:\d{2}:\d{2}$')
time_records = []
print(f"{'City':<12} {'Column':<20} {'Total':>10} {'Missing':>9} {'Bad fmt':>9} {'>=24h':>8}")
print("-" * 73)
for city in CITIES:
    st = data[city]["stop_times"]
    if st is None: continue
    for col in ["arrival_time", "departure_time"]:
        if col not in st.columns:
            print(f"{city:<12} {col:<20} {'[absent]':>10}")
            time_records.append({"city": city, "column": col, "total": None,
                "missing": None, "bad_format": None, "gte_24h": None, "note": "absent"})
            continue
        n_total   = len(st)
        n_missing = int(st[col].isnull().sum())
        non_null  = st[col].dropna().astype(str)
        bad_fmt   = non_null[~non_null.str.match(TIME_PATTERN)]
        n_bad     = len(bad_fmt)
        valid_fmt = non_null[non_null.str.match(TIME_PATTERN)]
        gte24     = int(valid_fmt.apply(lambda t: int(t.split(":")[0]) >= 24).sum())
        print(f"{city:<12} {col:<20} {n_total:>10,} {n_missing:>9,} {n_bad:>9,} {gte24:>8,}")
        time_records.append({"city": city, "column": col, "total": n_total,
            "missing": n_missing, "bad_format": n_bad, "gte_24h": gte24, "note": ""})
        if n_bad > 0:
            print(f"  sample bad values ({city}/{col}): {bad_fmt.head(5).tolist()}")
time_df = pd.DataFrame(time_records)

City         Column                    Total   Missing   Bad fmt    >=24h
-------------------------------------------------------------------------
ahmedabad    arrival_time            646,854         0         0    1,994
ahmedabad    departure_time          646,854         0         0    1,994
bengaluru    arrival_time          1,540,686         0         0        0
bengaluru    departure_time        1,540,686         0         0        0
davangere    arrival_time             17,783         0         0        0
davangere    departure_time           17,783         0         0        0
delhi        arrival_time          2,250,290         0         0    3,074
delhi        departure_time        2,250,290         0         0    3,074
hdbrt        arrival_time            194,059         0         0        8
hdbrt        departure_time          194,059         0         0        9
mumbai       arrival_time          2,655,026         0         0   14,274
mumbai       departure_time        2,6

### GTFS times — three critical rules

1. **Times can exceed 23:59:59** — `25:10:00` is intentional, not an error.
2. **Never use `pd.to_datetime()`** — convert to total seconds instead.
3. **Missing times are not errors** — agencies only record timepoint stops. Do **not** fill with `0`.

In [10]:
# Phase 2, Cell 5: Stop coordinate validation (India bounding box)
LAT_MIN, LAT_MAX = 6.0, 38.0
LON_MIN, LON_MAX = 68.0, 98.0
coord_records = []
print(f"{'City':<12} {'Stops':>8} {'miss lat':>10} {'miss lon':>10} {'bad lat':>9} {'bad lon':>9}")
print("-" * 64)
for city in CITIES:
    sdf = data[city]["stops"]
    if sdf is None: continue
    n = len(sdf)
    if "stop_lat" in sdf.columns:
        lat = pd.to_numeric(sdf["stop_lat"], errors="coerce")
        ml  = int(lat.isnull().sum()); bl = int(((lat < LAT_MIN) | (lat > LAT_MAX)).sum())
    else:
        lat = None; ml = bl = None
    if "stop_lon" in sdf.columns:
        lon = pd.to_numeric(sdf["stop_lon"], errors="coerce")
        mn  = int(lon.isnull().sum()); bn = int(((lon < LON_MIN) | (lon > LON_MAX)).sum())
    else:
        lon = None; mn = bn = None
    def fv(v): return f"{v:>9,}" if v is not None else "      N/A"
    print(f"{city:<12} {n:>8,} {fv(ml):>10} {fv(mn):>10} {fv(bl):>9} {fv(bn):>9}")
    coord_records.append({"city": city, "total_stops": n,
        "missing_lat": ml, "missing_lon": mn, "bad_lat": bl, "bad_lon": bn})
    if bl and bl > 0 and lat is not None:
        bm  = (lat < LAT_MIN) | (lat > LAT_MAX)
        sc  = [c for c in ["stop_id","stop_name","stop_lat","stop_lon"] if c in sdf.columns]
        print(f"  sample out-of-range ({city}):")
        print(sdf[bm][sc].head(3).to_string(index=False))
coord_df = pd.DataFrame(coord_records)

City            Stops   miss lat   miss lon   bad lat   bad lon
----------------------------------------------------------------
ahmedabad       3,235          0          0         0         0
bengaluru       9,960          0          0         0         0
davangere         146          0          0         0         0
delhi           4,192          0          0         0         0
hdbrt           2,332          0          0         0         0
mumbai         10,279          0          0         0         0
pune            6,745          0          0         0         0
rajkot            760          0          0         0         0


### Coordinate problems — what NOT to do

- `stop_lat = 0`: Do **not** include in maps. 0,0 is in the ocean.
- Lat/lon outside India: Do **not** silently drop; check whether the stop appears in stop_times.
- NaN coordinates: Do **not** fill with city-centre — this creates a false cluster.

In [11]:
# Phase 2, Cell 6: Calendar date range validation
cal_records = []
print(f"{'City':<12} {'Rows':>5}  {'start min':<14} {'start max':<14} {'end min':<14} {'end max':<14} {'end<start':>9}")
print("-" * 90)
for city in CITIES:
    cdf = data[city]["calendar"]
    if cdf is None: continue
    row = {"city": city, "rows": len(cdf)}; parsed = {}
    for col in ["start_date", "end_date"]:
        if col not in cdf.columns:
            row.update({f"{col}_min": "N/A", f"{col}_max": "N/A", f"{col}_parse_errors": None})
            parsed[col] = None; continue
        p = pd.to_datetime(cdf[col].astype(str), format="%Y%m%d", errors="coerce")
        parsed[col] = p
        row[f"{col}_min"]          = str(p.min().date()) if p.notna().any() else "N/A"
        row[f"{col}_max"]          = str(p.max().date()) if p.notna().any() else "N/A"
        row[f"{col}_parse_errors"] = int(p.isnull().sum())
    if parsed.get("start_date") is not None and parsed.get("end_date") is not None:
        both = parsed["start_date"].notna() & parsed["end_date"].notna()
        row["end_before_start"] = int((parsed["end_date"][both] < parsed["start_date"][both]).sum())
    else:
        row["end_before_start"] = None
    print(f"{row['city']:<12} {row['rows']:>5}  {str(row.get('start_date_min','N/A')):<14} {str(row.get('start_date_max','N/A')):<14} "
          f"{str(row.get('end_date_min','N/A')):<14} {str(row.get('end_date_max','N/A')):<14} {str(row.get('end_before_start','N/A')):>9}")
    cal_records.append(row)
cal_df_report = pd.DataFrame(cal_records)

City          Rows  start min      start max      end min        end max        end<start
------------------------------------------------------------------------------------------
ahmedabad        1  2026-09-30     2026-09-30     2027-03-29     2027-03-29             0
bengaluru        1  2026-09-07     2026-09-07     2027-09-07     2027-09-07             0
davangere        1  2026-06-02     2026-06-02     2026-11-29     2026-11-29             0
delhi            1  2021-09-01     2021-09-01     2022-10-01     2022-10-01             0
hdbrt            2  2026-08-24     2026-08-24     2027-02-20     2027-02-20             0
mumbai           1  2026-08-19     2026-08-19     2027-02-15     2027-02-15             0
pune             1  2026-10-07     2026-10-07     2027-04-05     2027-04-05             0
rajkot           1  2026-09-16     2026-09-16     2027-03-15     2027-03-15             0


### Calendar problems — what NOT to do

- `end_date < start_date`: definite data entry error. Do **not** swap dates without confirmation.
- Parse errors: value not readable as YYYYMMDD. Do **not** silently drop the row.
- Very wide date ranges: may be a placeholder. Do **not** assume the feed is current.

In [12]:
# Phase 2, Cell 7: Build city x issue summary and save
REPORTS_DIR = Path("reports")
REPORTS_DIR.mkdir(exist_ok=True)

def safe_int(val):
    try: return int(val)
    except (TypeError, ValueError): return None

summary_rows = []
for city in CITIES:
    row = {"city": city}
    cm  = missing_df[missing_df["city"] == city] if not missing_df.empty else pd.DataFrame()
    row["missing_columns_with_nulls"] = len(cm)
    row["missing_total_cells"]        = safe_int(cm["missing_count"].sum()) if not cm.empty else 0
    cd  = dup_df[(dup_df["city"] == city) & (dup_df["check"] == "duplicate_rows")]
    row["duplicate_rows"] = safe_int(cd["count"].sum())
    cd2 = dup_df[(dup_df["city"] == city) & (dup_df["check"].isin(["duplicate_route_id","duplicate_trip_id","duplicate_stop_id"]))]
    row["duplicate_ids"]  = safe_int(cd2["count"].sum())
    cx  = xref_df[xref_df["city"] == city]
    for cc, label in [("route_id","orphan_trip_route_id"),("trip_id","orphan_stoptime_trip_id"),
                      ("stop_id","orphan_stoptime_stop_id"),("service_id","orphan_trip_service_id")]:
        mask = cx["child_col"] == cc
        row[label] = safe_int(cx.loc[mask, "orphan_count"].sum()) if mask.any() else 0
    ct  = time_df[time_df["city"] == city] if not time_df.empty else pd.DataFrame()
    row["time_missing"]    = safe_int(ct["missing"].sum())    if not ct.empty else 0
    row["time_bad_format"] = safe_int(ct["bad_format"].sum()) if not ct.empty else 0
    row["time_gte_24h"]    = safe_int(ct["gte_24h"].sum())    if not ct.empty else 0
    cc2 = coord_df[coord_df["city"] == city] if not coord_df.empty else pd.DataFrame()
    if not cc2.empty:
        row["coord_missing_lat"] = safe_int(cc2["missing_lat"].iloc[0])
        row["coord_missing_lon"] = safe_int(cc2["missing_lon"].iloc[0])
        row["coord_bad_lat"]     = safe_int(cc2["bad_lat"].iloc[0])
        row["coord_bad_lon"]     = safe_int(cc2["bad_lon"].iloc[0])
    else:
        row.update({k: None for k in ["coord_missing_lat","coord_missing_lon","coord_bad_lat","coord_bad_lon"]})
    ck  = cal_df_report[cal_df_report["city"] == city] if not cal_df_report.empty else pd.DataFrame()
    if not ck.empty:
        row["cal_parse_errors"]     = safe_int((ck["start_date_parse_errors"].iloc[0] or 0) + (ck["end_date_parse_errors"].iloc[0] or 0))
        row["cal_end_before_start"] = safe_int(ck["end_before_start"].iloc[0])
    else:
        row.update({"cal_parse_errors": None, "cal_end_before_start": None})
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)
out_path   = REPORTS_DIR / "data_quality_checks.csv"
summary_df.to_csv(out_path, index=False)
print(f"Saved to: {out_path}")
SEP = "=" * 115
print(f"\n{SEP}\n  PHASE 2 SUMMARY -- Data Quality Checks (city x issue)\n{SEP}")
pd.set_option("display.max_columns", None); pd.set_option("display.width", 120)
print(summary_df.to_string(index=False))
print(SEP)

Saved to: reports\data_quality_checks.csv

  PHASE 2 SUMMARY -- Data Quality Checks (city x issue)
     city  missing_columns_with_nulls  missing_total_cells  duplicate_rows  duplicate_ids  orphan_trip_route_id  orphan_stoptime_trip_id  orphan_stoptime_stop_id  orphan_trip_service_id  time_missing  time_bad_format  time_gte_24h  coord_missing_lat  coord_missing_lon  coord_bad_lat  coord_bad_lon  cal_parse_errors  cal_end_before_start
ahmedabad                           1                14323               0              0                     0                        0                        0                       0             0                0          3988                  0                  0              0              0                 0                     0
bengaluru                           5                30368               0              0                     0                        0                        0                       0             0                0       

### Phase 2 summary legend

| Column | Healthy | Action if non-zero |
|--------|:-------:|---|
| `missing_total_cells` | 0 | Document; decide handling in Phase 3 |
| `duplicate_rows` / `duplicate_ids` | 0 | Investigate; flag for mentor before dropping |
| `orphan_*` columns | 0 | Those rows will lose data in joins |
| `time_bad_format` | 0 | Fix or exclude before time features |
| `time_gte_24h` | any | **Normal** — overnight trips; not an error |
| `coord_bad_*` | 0 | Exclude from spatial analysis |
| `cal_end_before_start` | 0 | Definite data entry error; flag for review |

---
# Phase 3: Time-Based Feature Engineering

**Goal:** Derive actionable time features from `stop_times` and service-day features from `calendar`.

New columns created:
- `arr_seconds`, `dep_seconds` — times as total seconds from midnight (handles HH ≥ 24)
- `dep_hour_raw` — raw hour (can be 24+), `dep_hour` — normalised 0–23, `dep_minute`
- `next_day_flag` — True when the trip crosses midnight
- `time_period` — morning / afternoon / evening / night
- `day_type` and `operating_days` on calendar, then joined to trips

Processing is done **city by city** to avoid loading all stop_times into memory at once.

In [13]:
# Phase 3, Cell 1: Helper function -- convert GTFS time string to seconds
#
# GTFS times look like HH:MM:SS.  HH can be 24 or more for overnight trips.
# We split on ':' and compute  h*3600 + m*60 + s  using plain arithmetic.
# Invalid or missing values return float('nan') -- they are COUNTED, not dropped.

def gtfs_to_seconds(time_str):
    """
    Convert a GTFS time string HH:MM:SS to total seconds since midnight.
    Hours can be 24 or more (e.g. 25:10:00 for overnight trips).
    Returns float('nan') for any invalid or missing value.
    """
    # Handle genuine NaN / None values
    try:
        if pd.isna(time_str):
            return float('nan')
    except TypeError:
        pass  # pd.isna raises TypeError on some non-scalar types

    try:
        parts = str(time_str).strip().split(':')
        if len(parts) != 3:
            return float('nan')         # wrong number of colon-separated parts
        h, m, s = int(parts[0]), int(parts[1]), int(parts[2])
        if not (0 <= m <= 59 and 0 <= s <= 59):
            return float('nan')         # minutes or seconds out of range
        return float(h * 3600 + m * 60 + s)
    except (ValueError, AttributeError):
        return float('nan')             # non-numeric parts


# Quick test -- verify the function handles all expected cases correctly
import math

test_cases = [
    ('25:10:00', 25*3600 + 10*60),   # overnight trip  -> 90600 seconds
    ('08:30:00',  8*3600 + 30*60),   # normal morning  -> 30600 seconds
    ('00:00:00',  0.0),              # midnight start  -> 0
    ('23:59:59',  23*3600+59*60+59), # last second of normal day
    ('',          float('nan')),     # empty string    -> NaN
    (None,        float('nan')),     # Python None     -> NaN
    ('bad_value', float('nan')),     # invalid format  -> NaN
    ('08:99:00',  float('nan')),     # minutes out of range -> NaN
]

print("gtfs_to_seconds function test:")
all_ok = True
for t_in, expected in test_cases:
    result = gtfs_to_seconds(t_in)
    both_nan = math.isnan(result) and math.isnan(float(str(expected)))
    match    = both_nan or (result == expected)
    status   = 'OK' if match else 'FAIL'
    if not match: all_ok = False
    print(f"  gtfs_to_seconds({str(t_in)!r:<14}) = {str(result):>8}  {status}")
print(f"\nAll tests passed: {all_ok}")

gtfs_to_seconds function test:
  gtfs_to_seconds('25:10:00'    ) =  90600.0  OK
  gtfs_to_seconds('08:30:00'    ) =  30600.0  OK
  gtfs_to_seconds('00:00:00'    ) =      0.0  OK
  gtfs_to_seconds('23:59:59'    ) =  86399.0  OK
  gtfs_to_seconds(''            ) =      nan  OK
  gtfs_to_seconds('None'        ) =      nan  OK
  gtfs_to_seconds('bad_value'   ) =      nan  OK
  gtfs_to_seconds('08:99:00'    ) =      nan  OK

All tests passed: True


### How `25:10:00` is handled — step by step

```
Input string     : '25:10:00'
After split(':') : ['25', '10', '00']  ->  h=25, m=10, s=0
dep_seconds      : 25*3600 + 10*60 + 0 = 90,600 seconds
dep_hour_raw     : 90600 // 3600 = 25          (raw hour, may be >= 24)
dep_hour         : 25 % 24       =  1          (normalised to 0-23 clock hour)
dep_minute       : (90600 % 3600) // 60 = 10
next_day_flag    : True                        (because dep_hour_raw >= 24)
time_period      : 'night'                     (dep_hour = 1, falls in 0-4 range)
```

**Why this matters:** A bus departing at `25:10:00` and the next bus at `26:00:00` are
50 minutes apart. If we naively wrap to 1:10 AM and 2:00 AM we still get 50 minutes,
but if we mixed 25:10:00 with a midnight-wrapping parser we could get 16+ hours.
Using `dep_seconds` for headway calculations always gives the correct gap.

**`next_day_flag`** is the marker that tells downstream code: this stop-time event
occurred on the next calendar day, even though it belongs to the current service day.

In [14]:
# Phase 3, Cell 2: Time period classification function
#
# We classify using dep_hour (0-23, the NORMALISED hour) so that
# an overnight trip with dep_hour_raw=25 gets dep_hour=1 and is correctly
# labelled 'night' rather than being left as an outlier.

def assign_time_period(dep_hour):
    """
    Assign a time-period label based on the normalised departure hour (0-23).

    Rules:
      morning   05:00 - 11:59   dep_hour  5 to 11
      afternoon 12:00 - 16:59   dep_hour 12 to 16
      evening   17:00 - 20:59   dep_hour 17 to 20
      night     21:00 - 04:59   dep_hour 21-23 or 0-4  (wraps across midnight)

    Returns float('nan') if dep_hour is NaN.
    """
    if pd.isna(dep_hour):
        return float('nan')
    h = int(dep_hour)
    if  5 <= h <= 11: return 'morning'
    if 12 <= h <= 16: return 'afternoon'
    if 17 <= h <= 20: return 'evening'
    return 'night'   # covers 0-4 and 21-23


# Verify all hour values map correctly
print("Time period mapping (hour -> period):")
for h in range(24):
    period = assign_time_period(h)
    print(f"  hour {h:>2d}  ->  {period}")

Time period mapping (hour -> period):
  hour  0  ->  night
  hour  1  ->  night
  hour  2  ->  night
  hour  3  ->  night
  hour  4  ->  night
  hour  5  ->  morning
  hour  6  ->  morning
  hour  7  ->  morning
  hour  8  ->  morning
  hour  9  ->  morning
  hour 10  ->  morning
  hour 11  ->  morning
  hour 12  ->  afternoon
  hour 13  ->  afternoon
  hour 14  ->  afternoon
  hour 15  ->  afternoon
  hour 16  ->  afternoon
  hour 17  ->  evening
  hour 18  ->  evening
  hour 19  ->  evening
  hour 20  ->  evening
  hour 21  ->  night
  hour 22  ->  night
  hour 23  ->  night


### Time period definitions

| Period | `dep_hour` range | Hours covered |
|--------|:---:|---|
| `morning` | 5 – 11 | 05:00 to 11:59 |
| `afternoon` | 12 – 16 | 12:00 to 16:59 |
| `evening` | 17 – 20 | 17:00 to 20:59 |
| `night` | 0 – 4 and 21 – 23 | 00:00–04:59 and 21:00–23:59 |

We use `dep_hour` (not `dep_hour_raw`) so that all overnight trips (e.g. `dep_hour_raw = 25`
gives `dep_hour = 1`) fall into the correct period without extra special-casing.

`next_day_flag` is preserved separately for headway and scheduling calculations.

In [15]:
# Phase 3, Cell 3: Apply time features to stop_times for every city
# Processed city-by-city to avoid holding all cities in memory simultaneously.

time_feature_log = []   # per-city stats for the summary cell

for city in CITIES:
    st = data[city]['stop_times']
    if st is None:
        print(f"  {city}: stop_times not available, skipping.")
        continue

    n_rows = len(st)

    # ---------- arrival_time -> arr_seconds ----------------------------------
    if 'arrival_time' in st.columns:
        st['arr_seconds'] = st['arrival_time'].apply(gtfs_to_seconds)
        # Bad-format count = new NaNs beyond the ones already in the raw column
        n_arr_bad = max(0, int(st['arr_seconds'].isna().sum()) - int(st['arrival_time'].isna().sum()))
    else:
        st['arr_seconds'] = float('nan')
        n_arr_bad = 0

    # ---------- departure_time -> dep_seconds --------------------------------
    if 'departure_time' in st.columns:
        n_orig_missing   = int(st['departure_time'].isna().sum())
        st['dep_seconds'] = st['departure_time'].apply(gtfs_to_seconds)
        n_dep_bad = max(0, int(st['dep_seconds'].isna().sum()) - n_orig_missing)
    else:
        st['dep_seconds'] = float('nan')
        n_dep_bad = 0

    # ---------- hour and minute columns -------------------------------------
    # dep_hour_raw : raw hour, may be >= 24 for overnight trips
    st['dep_hour_raw'] = st['dep_seconds'] // 3600

    # dep_hour : normalised to 0-23 (the wall-clock hour)
    st['dep_hour'] = st['dep_hour_raw'] % 24

    # dep_minute : minute within the hour
    st['dep_minute'] = (st['dep_seconds'] % 3600) // 60

    # ---------- next_day_flag -----------------------------------------------
    # True when the scheduled time crosses midnight relative to service day start
    st['next_day_flag'] = st['dep_hour_raw'] >= 24

    # ---------- time_period --------------------------------------------------
    st['time_period'] = st['dep_hour'].apply(assign_time_period)

    # Store back
    data[city]['stop_times'] = st

    n_next_day  = int(st['next_day_flag'].sum())
    n_time_nan  = int(st['time_period'].isna().sum())

    print(f"  {city:<12}: {n_rows:>9,} rows | "
          f"arr_bad={n_arr_bad:>6,} | dep_bad={n_dep_bad:>6,} | "
          f"next_day={n_next_day:>7,} | time_period_NaN={n_time_nan:>6,}")

    time_feature_log.append({'city': city, 'rows': n_rows,
        'arr_bad_format': n_arr_bad, 'dep_bad_format': n_dep_bad,
        'next_day_count': n_next_day, 'time_period_nan': n_time_nan})

print("\nTime-based columns added to stop_times for all cities.")
print("New columns: arr_seconds, dep_seconds, dep_hour_raw, dep_hour, dep_minute, next_day_flag, time_period")

  ahmedabad   :   646,854 rows | arr_bad=     0 | dep_bad=     0 | next_day=  1,994 | time_period_NaN=     0
  bengaluru   : 1,540,686 rows | arr_bad=     0 | dep_bad=     0 | next_day=      0 | time_period_NaN=     0
  davangere   :    17,783 rows | arr_bad=     0 | dep_bad=     0 | next_day=      0 | time_period_NaN=     0
  delhi       : 2,250,290 rows | arr_bad=     0 | dep_bad=     0 | next_day=  3,074 | time_period_NaN=     0
  hdbrt       :   194,059 rows | arr_bad=     0 | dep_bad=     0 | next_day=      9 | time_period_NaN=     0
  mumbai      : 2,655,026 rows | arr_bad=     0 | dep_bad=     0 | next_day= 14,274 | time_period_NaN=     0
  pune        :   429,302 rows | arr_bad=     0 | dep_bad=     0 | next_day=    179 | time_period_NaN=     0
  rajkot      :    94,809 rows | arr_bad=     0 | dep_bad=     0 | next_day=      0 | time_period_NaN=     0

Time-based columns added to stop_times for all cities.
New columns: arr_seconds, dep_seconds, dep_hour_raw, dep_hour, dep_minut

In [16]:
# Phase 3, Cell 4: Create day_type and operating_days from calendar, attach to trips
#
# GTFS calendar has one row per service_id with binary columns for each day of the week.
# We classify each service into a day_type, then join to trips via service_id.

WEEKDAY_COLS = ['monday', 'tuesday', 'wednesday', 'thursday', 'friday']
WEEKEND_COLS = ['saturday', 'sunday']


def classify_day_type(row):
    """
    Classify a calendar row into a day_type string.

    weekday_only : at least one weekday = 1  AND  saturday = 0  AND  sunday = 0
    weekend_only : saturday or sunday = 1   AND  all weekdays = 0
    all_days     : all 7 days = 1
    mixed        : any other combination
    """
    wd_vals = [row.get(c, 0) for c in WEEKDAY_COLS]
    we_vals = [row.get(c, 0) for c in WEEKEND_COLS]
    has_wd  = any(v == 1 for v in wd_vals)
    has_we  = any(v == 1 for v in we_vals)
    no_wd   = all(v == 0 for v in wd_vals)
    no_we   = all(v == 0 for v in we_vals)

    if has_wd and no_we:  return 'weekday_only'
    if has_we and no_wd:  return 'weekend_only'
    if has_wd and has_we:
        all_cols = WEEKDAY_COLS + WEEKEND_COLS
        if all(row.get(c, 0) == 1 for c in all_cols):
            return 'all_days'
        return 'mixed'
    return 'mixed'   # neither group active (unusual feed)


def get_operating_days(row):
    """Return a comma-separated list of day names where value = 1."""
    active = [d for d in WEEKDAY_COLS + WEEKEND_COLS if row.get(d, 0) == 1]
    return ','.join(active) if active else 'none'


for city in CITIES:
    cal   = data[city]['calendar']
    trips = data[city]['trips']
    if cal is None or trips is None:
        print(f"  {city}: calendar or trips missing, skipping.")
        continue

    # Check which day columns actually exist in this city's calendar file
    avail_wd = [c for c in WEEKDAY_COLS if c in cal.columns]
    avail_we = [c for c in WEEKEND_COLS if c in cal.columns]

    if not avail_wd and not avail_we:
        print(f"  {city}: no day-of-week columns in calendar, marking as unknown.")
        cal['day_type']       = 'unknown'
        cal['operating_days'] = 'unknown'
    else:
        cal['day_type']       = cal.apply(classify_day_type, axis=1)
        cal['operating_days'] = cal.apply(get_operating_days, axis=1)

    # Build service_id -> day_type / operating_days lookup
    lookup = cal.set_index('service_id')[['day_type', 'operating_days']]

    # Left-join onto trips: keeps ALL trips; unmatched get NaN
    trips = trips.merge(lookup, on='service_id', how='left')

    n_nan = int(trips['day_type'].isna().sum())
    vc    = trips['day_type'].value_counts().to_dict()
    print(f"  {city:<12}: {len(trips):>8,} trips | day_type NaN={n_nan:>5,} | counts={vc}")

    data[city]['calendar'] = cal
    data[city]['trips']    = trips

print("\nday_type and operating_days added to calendar and trips for all cities.")

  ahmedabad   :   14,323 trips | day_type NaN=    0 | counts={'all_days': 14323}
  bengaluru   :   57,836 trips | day_type NaN=    0 | counts={'all_days': 57836}
  davangere   :      586 trips | day_type NaN=    0 | counts={'all_days': 586}
  delhi       :   50,355 trips | day_type NaN=    0 | counts={'all_days': 50355}
  hdbrt       :    8,669 trips | day_type NaN=    0 | counts={'all_days': 8652, 'mixed': 17}
  mumbai      :   99,080 trips | day_type NaN=    0 | counts={'all_days': 99080}
  pune        :   10,128 trips | day_type NaN=    0 | counts={'all_days': 10128}
  rajkot      :    3,192 trips | day_type NaN=    0 | counts={'all_days': 3192}

day_type and operating_days added to calendar and trips for all cities.


### `day_type` classification logic

| `day_type` | Condition on calendar row |
|------------|---------------------------|
| `weekday_only` | At least one of Mon–Fri = 1, AND Sat = 0, AND Sun = 0 |
| `weekend_only` | Sat = 1 or Sun = 1, AND all of Mon–Fri = 0 |
| `all_days` | All 7 days = 1 |
| `mixed` | Any other combination (e.g. Mon–Sat but not Sun) |

`day_type` is computed from `calendar`, then **left-joined** to `trips` on `service_id`.

A `NaN` in `trips.day_type` means the trip’s `service_id` had no matching row in `calendar`
(an orphan ID flagged in Phase 2). We do **not** fill these NaNs with a default value.

`operating_days` is a comma-separated string of active day names (e.g. `monday,tuesday,wednesday,thursday,friday`).
It preserves the raw information for `mixed` services that need closer inspection.

In [17]:
# Phase 3, Cell 5: Sample rows and value counts
SEP = "=" * 95
print(SEP + "\n  PHASE 3 SUMMARY\n" + SEP)

# A. 5 sample stop_times rows with new time features (Pune)
print("\nA. Sample stop_times rows with time features (Pune, 5 rows with valid dep_seconds):")
st_pune = data['pune']['stop_times']
sample_cols = [c for c in ['trip_id', 'departure_time', 'dep_seconds',
                            'dep_hour_raw', 'dep_hour', 'dep_minute',
                            'next_day_flag', 'time_period']
               if c in st_pune.columns]
valid_dep = st_pune[st_pune['dep_seconds'].notna()]
print(valid_dep[sample_cols].head(5).to_string(index=False))

# Show an overnight example if any city has one
for city in CITIES:
    st = data[city]['stop_times']
    if st is None or 'next_day_flag' not in st.columns: continue
    overnight = st[st['next_day_flag'] == True]
    if len(overnight) > 0:
        sc = [c for c in ['trip_id','departure_time','dep_seconds','dep_hour_raw',
                           'dep_hour','next_day_flag','time_period'] if c in overnight.columns]
        print(f"\n  Overnight trip example ({city}):")
        print(overnight[sc].head(1).to_string(index=False))
        break

# B. time_period value counts per city
print("\nB. time_period value counts (each row = one stop_time record)")
print(f"{'City':<12} {'morning':>10} {'afternoon':>10} {'evening':>10} {'night':>10} {'NaN':>8}")
print("-" * 65)
for city in CITIES:
    st = data[city]['stop_times']
    if st is None or 'time_period' not in st.columns: continue
    vc  = st['time_period'].value_counts()
    nan = int(st['time_period'].isna().sum())
    print(f"{city:<12} {vc.get('morning',0):>10,} {vc.get('afternoon',0):>10,} "
          f"{vc.get('evening',0):>10,} {vc.get('night',0):>10,} {nan:>8,}")

# C. day_type value counts per city (trips level)
print("\nC. day_type value counts (each row = one trip)")
print(f"{'City':<12} {'weekday_only':>14} {'weekend_only':>13} {'all_days':>10} {'mixed':>8} {'NaN':>8}")
print("-" * 70)
for city in CITIES:
    tdf = data[city]['trips']
    if tdf is None or 'day_type' not in tdf.columns: continue
    vc  = tdf['day_type'].value_counts()
    nan = int(tdf['day_type'].isna().sum())
    print(f"{city:<12} {vc.get('weekday_only',0):>14,} {vc.get('weekend_only',0):>13,} "
          f"{vc.get('all_days',0):>10,} {vc.get('mixed',0):>8,} {nan:>8,}")

# D. Sample trips rows with day_type (Bengaluru)
print("\nD. Sample trips rows with day_type (Bengaluru, 5 rows):")
tdf_b    = data['bengaluru']['trips']
trip_cols = [c for c in ['trip_id','route_id','service_id','day_type','operating_days']
             if c in tdf_b.columns]
print(tdf_b[trip_cols].head(5).to_string(index=False))

print("\n" + SEP)

  PHASE 3 SUMMARY

A. Sample stop_times rows with time features (Pune, 5 rows with valid dep_seconds):
       trip_id departure_time  dep_seconds  dep_hour_raw  dep_hour  dep_minute  next_day_flag time_period
pune_trip_18_0       08:30:00      30600.0           8.0       8.0        30.0          False     morning
pune_trip_18_0       08:31:00      30660.0           8.0       8.0        31.0          False     morning
pune_trip_18_0       08:33:09      30789.0           8.0       8.0        33.0          False     morning
pune_trip_18_0       08:34:09      30849.0           8.0       8.0        34.0          False     morning
pune_trip_18_0       08:35:09      30909.0           8.0       8.0        35.0          False     morning

  Overnight trip example (ahmedabad):
              trip_id departure_time  dep_seconds  dep_hour_raw  dep_hour  next_day_flag time_period
ahmedabad_AMC_9351367       31:46:00     114360.0          31.0       7.0           True     morning

B. time_period valu

### Phase 3 summary — what was created

**Added to `stop_times` for every city:**

| New column | Dtype | What it represents |
|------------|:-----:|--------------------|
| `arr_seconds` | float / NaN | `arrival_time` as total seconds from midnight; NaN if invalid |
| `dep_seconds` | float / NaN | `departure_time` as total seconds from midnight; NaN if invalid |
| `dep_hour_raw` | float / NaN | Raw hour: `dep_seconds // 3600`, can be ≥ 24 |
| `dep_hour` | float / NaN | Normalised hour: `dep_hour_raw % 24`, always 0–23 |
| `dep_minute` | float / NaN | Minute: `(dep_seconds % 3600) // 60` |
| `next_day_flag` | bool | `True` when `dep_hour_raw >= 24` (overnight trip) |
| `time_period` | str / NaN | morning / afternoon / evening / night (based on `dep_hour`) |

**Added to `calendar` for every city:**

| New column | What it represents |
|------------|--------------------|
| `day_type` | weekday_only / weekend_only / all_days / mixed |
| `operating_days` | Comma-separated list of day names where value = 1 |

**Added to `trips` for every city (joined from calendar):**

| New column | What it represents |
|------------|--------------------|
| `day_type` | Inherited from calendar via left join on `service_id` |
| `operating_days` | Inherited from calendar |

> NaN in `trips.day_type` = orphan `service_id` (flagged in Phase 2). Do **not** fill with a default.

---
**Phase 3 complete.**

---
# Phase 4: Route-Level Feature Aggregation (per city)

**Goal:** Summarise transit operations at the route level for every city.

Key route features to build:
1. **`trip_count`**: Number of distinct scheduled trips operating on this route (`nunique()` on `trip_id`).
2. **`stop_count`**: Number of distinct physical stops served by this route (`nunique()` on `stop_id`).

> **Crucial Rule (Rule 3):** A single trip has many rows in `stop_times` (one row per stop).
> We must **never** count rows to determine the number of trips. We always use `.nunique()` on `trip_id`.

In [18]:
# Phase 4, Cell 1: Proof of why counting rows is WRONG
#
# We select one sample route from Pune to demonstrate the huge difference
# between raw row count in stop_times vs actual distinct trips.

# 1. Pick a city and a route that has trips
proof_city = 'pune'
sample_trips = data[proof_city]['trips']
sample_stop_times = data[proof_city]['stop_times']

# Pick the first route_id from the trips table
sample_route_id = sample_trips['route_id'].iloc[0]

# 2. Find all trip_ids belonging to this route
route_trip_ids = sample_trips[sample_trips['route_id'] == sample_route_id]['trip_id']

# 3. Count distinct trips for this route
distinct_trips = route_trip_ids.nunique()

# 4. Filter stop_times for these trip_ids
route_stop_times = sample_stop_times[sample_stop_times['trip_id'].isin(route_trip_ids)]

# 5. Count total rows in stop_times vs distinct stops
total_st_rows = len(route_stop_times)
distinct_stops = route_stop_times['stop_id'].nunique()

# 6. Display the proof comparison
print("=" * 65)
print("  PROOF: Why counting rows in stop_times gives WRONG results")
print("=" * 65)
print(f"  City                       : {proof_city.title()}")
print(f"  Sample Route ID            : {sample_route_id}")
print(f"  Distinct scheduled trips   : {distinct_trips:>6,}  <-- TRUE trip count (nunique)")
print(f"  Distinct stops on route    : {distinct_stops:>6,}  <-- UNIQUE physical stops")
print(f"  Raw rows in stop_times     : {total_st_rows:>6,}  <-- STOP VISITS, NOT TRIPS!")
print(f"  Inflation factor           : {total_st_rows / distinct_trips:.1f}x  (rows per trip on average)")
print("-" * 65)
print("  CONCLUSION: If we counted rows, we would report that this route")
print(f"  ran {total_st_rows} trips when in reality it only ran {distinct_trips} trips!")
print("=" * 65)

  PROOF: Why counting rows in stop_times gives WRONG results
  City                       : Pune
  Sample Route ID            : pune_18
  Distinct scheduled trips   :      3  <-- TRUE trip count (nunique)
  Distinct stops on route    :     29  <-- UNIQUE physical stops
  Raw rows in stop_times     :     87  <-- STOP VISITS, NOT TRIPS!
  Inflation factor           : 29.0x  (rows per trip on average)
-----------------------------------------------------------------
  CONCLUSION: If we counted rows, we would report that this route
  ran 87 trips when in reality it only ran 3 trips!


### What this proof proves

In GTFS:
- **`trips`** represents one bus run from start to terminus.
- **`stop_times`** has **one row per stop visited** on that trip.

If a bus visits 35 stops along its journey:
$$\text{1 trip} = 35 \text{ rows in stop\_times}$$

If that bus route runs 20 times a day, `stop_times` will contain $20 \times 35 = 700$ rows!

- Counting rows (`len(df)` or `.count()`) counts **stop-arrival events**, not trips.
- Counting distinct trip IDs (`df['trip_id'].nunique()`) gives the **true scheduled trip count**.

> Always use `.nunique()` on `trip_id` when measuring service volume.

In [19]:
# Phase 4, Cell 2: Aggregate route features per city
#
# For each city:
#   1. trip_count: distinct trips per route from trips (nunique)
#   2. stop_count: distinct stops per route from stop_times (nunique)
#      (joined efficiently by selecting only needed columns first)
#   3. Merge onto routes table with reset_index()
#   4. Save outputs to processed/ folder (Rule 1)

from pathlib import Path
PROCESSED_DIR = Path("processed")
PROCESSED_DIR.mkdir(exist_ok=True)

route_features = {}       # dictionary: route_features[city] = DataFrame
all_routes_list = []      # collector to create a combined cross-city dataset

for city in CITIES:
    routes_df = data[city]['routes']
    trips_df  = data[city]['trips']
    st_df     = data[city]['stop_times']

    if routes_df is None or trips_df is None:
        print(f"  {city:<12}: routes or trips missing, skipping.")
        continue

    # -------------------------------------------------------------------------
    # Step 1: Compute trip_count per route
    # Group trips by route_id and count unique trip_id values
    # -------------------------------------------------------------------------
    trip_counts = (
        trips_df.groupby('route_id')['trip_id']
        .nunique()
        .reset_index(name='trip_count')
    )

    # -------------------------------------------------------------------------
    # Step 2: Compute stop_count per route (EFFICIENT JOIN)
    # We slice ONLY needed columns before joining: ['route_id', 'trip_id'] and ['trip_id', 'stop_id'].
    # This prevents loading millions of unused text columns into memory during join.
    # -------------------------------------------------------------------------
    if st_df is not None and 'stop_id' in st_df.columns:
        trips_sub = trips_df[['route_id', 'trip_id']].dropna()
        st_sub    = st_df[['trip_id', 'stop_id']].dropna()

        # Inner join links each stop event directly to its route
        merged_route_stops = trips_sub.merge(st_sub, on='trip_id', how='inner')

        # Count distinct physical stops served per route
        stop_counts = (
            merged_route_stops.groupby('route_id')['stop_id']
            .nunique()
            .reset_index(name='stop_count')
        )
    else:
        stop_counts = pd.DataFrame(columns=['route_id', 'stop_count'])

    # -------------------------------------------------------------------------
    # Step 3: Combine metrics with base routes DataFrame
    # Keep core descriptive route columns if present
    # -------------------------------------------------------------------------
    base_cols = ['city', 'route_id']
    for col in ['route_short_name', 'route_long_name', 'route_type']:
        if col in routes_df.columns:
            base_cols.append(col)

    city_route_feats = routes_df[base_cols].copy()

    # Left join ensures every route in routes.txt is preserved
    city_route_feats = city_route_feats.merge(trip_counts, on='route_id', how='left')
    city_route_feats = city_route_feats.merge(stop_counts, on='route_id', how='left')

    # Store in memory
    route_features[city] = city_route_feats
    all_routes_list.append(city_route_feats)

    # Save per-city route features to processed/ directory (Rule 1)
    city_out_file = PROCESSED_DIR / f"{city}_route_features.csv"
    city_route_feats.to_csv(city_out_file, index=False)

    n_routes = len(city_route_feats)
    has_trips = int(city_route_feats['trip_count'].notna().sum())
    print(f"  {city:<12}: {n_routes:>6,} routes | {has_trips:>6,} have scheduled trips | saved -> {city_out_file.name}")

# Combine all cities into a single master route features file
all_route_features = pd.concat(all_routes_list, ignore_index=True)
master_out_file = PROCESSED_DIR / "all_cities_route_features.csv"
all_route_features.to_csv(master_out_file, index=False)
print(f"\nSaved master route features to: {master_out_file} ({len(all_route_features):,} total routes across 8 cities)")

  ahmedabad   :    978 routes |    972 have scheduled trips | saved -> ahmedabad_route_features.csv
  bengaluru   :  4,434 routes |  4,416 have scheduled trips | saved -> bengaluru_route_features.csv
  davangere   :     93 routes |     14 have scheduled trips | saved -> davangere_route_features.csv
  delhi       :  1,270 routes |  1,187 have scheduled trips | saved -> delhi_route_features.csv
  hdbrt       :    632 routes |    630 have scheduled trips | saved -> hdbrt_route_features.csv
  mumbai      :  1,609 routes |  1,115 have scheduled trips | saved -> mumbai_route_features.csv
  pune        :    628 routes |    628 have scheduled trips | saved -> pune_route_features.csv
  rajkot      :    271 routes |    271 have scheduled trips | saved -> rajkot_route_features.csv

Saved master route features to: processed\all_cities_route_features.csv (9,915 total routes across 8 cities)


### Step-by-step breakdown of the aggregation logic

1. **`trips_df.groupby('route_id')['trip_id'].nunique()`**:
   - Groups trips by `route_id` and counts only distinct `trip_id`s.
   - Returns a Series indexed by `route_id`.
2. **`.reset_index(name='trip_count')`**:
   - Turns `route_id` back into a regular DataFrame column and names the aggregated metric column `trip_count`.
3. **Memory-efficient join (`trips_sub.merge(st_sub)`)**:
   - Slicing `trips[['route_id', 'trip_id']]` and `stop_times[['trip_id', 'stop_id']]` drops all bulky time and text columns.
   - Keeps RAM consumption low and makes inner join fast on millions of rows.
4. **`.nunique()` on `stop_id`**:
   - A route may visit the same physical stop multiple times (loop routes or return trips).
   - `nunique()` ensures we count distinct physical stops served by the route.
5. **Left join onto `routes`**:
   - Preserves all routes declared by the transit agency. Routes with no trips in `trips.txt` retain `NaN`, documenting the absence of scheduled service without altering raw truth.

In [20]:
# Phase 4, Cell 3: Top 5, Bottom 5, and Summary Statistics per city
#
# Print describe() and inspect extremes in trip frequency and route size.

SEP = "=" * 95
print(SEP)
print("  PHASE 4 ROUTE AGGREGATION SUMMARY -- Top/Bottom Routes & Statistics")
print(SEP)

for city in CITIES:
    rf = route_features.get(city)
    if rf is None: continue

    print(f"\n{'#' * 35} {city.upper()} {'#' * 35}")

    # 1. Summary statistics (describe)
    print("\n--- A. Route Metrics Summary Statistics (describe) ---")
    desc = rf[['trip_count', 'stop_count']].describe().round(1)
    print(desc.to_string())

    # Select columns to display for top/bottom tables
    disp_cols = ['route_id']
    if 'route_short_name' in rf.columns: disp_cols.append('route_short_name')
    if 'route_long_name' in rf.columns:  disp_cols.append('route_long_name')
    disp_cols.extend(['trip_count', 'stop_count'])

    # Valid routes with trip_count not null
    valid_rf = rf[rf['trip_count'].notna()]

    # 2. Top 5 routes by trip_count
    print("\n--- B. Top 5 Most Frequent Routes (by trip_count) ---")
    top5 = valid_rf.sort_values(by='trip_count', ascending=False).head(5)
    print(top5[disp_cols].to_string(index=False))

    # 3. Bottom 5 routes by trip_count
    print("\n--- C. Bottom 5 Least Frequent Routes (by trip_count) ---")
    bottom5 = valid_rf.sort_values(by='trip_count', ascending=True).head(5)
    print(bottom5[disp_cols].to_string(index=False))

print("\n" + SEP)

  PHASE 4 ROUTE AGGREGATION SUMMARY -- Top/Bottom Routes & Statistics

################################### AHMEDABAD ###################################

--- A. Route Metrics Summary Statistics (describe) ---
       trip_count  stop_count
count       972.0       972.0
mean         14.7        43.1
std          20.3        22.3
min           1.0         4.0
25%           4.0        25.0
50%           8.0        40.0
75%          16.0        58.0
max         134.0       117.0

--- B. Top 5 Most Frequent Routes (by trip_count) ---
          route_id route_short_name                                route_long_name  trip_count  stop_count
ahmedabad_AMC_3054            9U-VM   Gota Vasant Nagar Township to Maninagar (UP)       134.0        37.0
ahmedabad_AMC_3044            9D-MV Maninagar to Gota Vasant Nagar Township (DOWN)       130.0        40.0
ahmedabad_AMC_2952               1U                    Ghuma Gam to Maninagar (UP)       128.0        34.0
ahmedabad_AMC_2945               1D   

### What the summary statistics and route extremes reveal

1. **Wide variance in `trip_count`**:
   - In major networks like Bengaluru and Mumbai, high-frequency trunk lines operate hundreds of trips daily.
   - Conversely, bottom routes often operate only 1 or 2 trips daily. These are typically:
     - Peak-hour only feeder routes
     - School/factory special shuttles
     - Early morning or late night depot repositioning trips

2. **Stop count distributions**:
   - Express / BRT lines typically serve fewer distinct stops (concentrated at major interchanges).
   - Suburban or local stage carriages serve 40 to 80+ distinct stops.

3. **Outputs Saved (Rule 1)**:
   - `processed/{city}_route_features.csv` (per-city route metrics)
   - `processed/all_cities_route_features.csv` (combined dataset across all 8 cities)

---
**Phase 4 complete.**

---
# Phase 5: Frequency and Headway Feature Engineering

**Goal:** Understand service cadence, regularity, and passenger waiting intervals per route.

### Core Concepts Defined:
1. **Trip Count**: The total number of distinct scheduled trips operating over the entire service day.
2. **Frequency (`trips_per_hour`)**: How many scheduled trips depart within a specific 1-hour window (e.g. 08:00–08:59).
3. **Headway (`headway_min`)**: The time gap in minutes between consecutive departures on the same route, in the same direction, on the same day type.

### Concrete Example:
Suppose Route 101 has morning departures at **08:00**, **08:15**, and **08:30**:
- **Trip count** in this window = 3 trips
- **Frequency** = 3 trips / hour (between 08:00 and 08:59)
- **Headway** = 15 minutes (08:15 − 08:00 = 15m; 08:30 − 08:15 = 15m)

> **Why Headway is Crucial:** Two routes can both run 20 trips a day. But Route A might run every 30 minutes like clockwork (reliable, short wait), while Route B runs in irregular batches with 90-minute gaps (long waits, bunching). Headway captures the real rider experience!

In [21]:
# Phase 5, Cell 1: Extract trip start time from first stop
#
# For each trip, the true start time is the departure_time at the lowest stop_sequence.
# We extract this from stop_times and attach it directly to trips.

for city in CITIES:
    st = data[city]['stop_times']
    trips = data[city]['trips']

    if st is None or trips is None:
        continue

    # 1. Filter to rows with valid dep_seconds
    valid_st = st[st['dep_seconds'].notna()].copy()

    # 2. Ensure stop_sequence is numeric for proper sorting
    valid_st['stop_sequence_num'] = pd.to_numeric(valid_st['stop_sequence'], errors='coerce')

    # 3. Sort by trip_id and stop_sequence ascending, then keep the very first stop
    first_stops = (
        valid_st.sort_values(['trip_id', 'stop_sequence_num'])
        .drop_duplicates(subset=['trip_id'], keep='first')
    )

    # 4. Select trip start features
    start_cols = ['trip_id', 'dep_seconds', 'dep_hour', 'dep_minute', 'time_period']
    trip_starts = first_stops[start_cols].copy()
    trip_starts.rename(columns={
        'dep_seconds': 'start_seconds',
        'dep_hour':    'start_hour',
        'dep_minute':  'start_minute',
        'time_period': 'start_time_period'
    }, inplace=True)

    # 5. Merge start time onto trips table
    # Drop existing start columns if re-running cell
    drop_existing = [c for c in trip_starts.columns if c != 'trip_id' and c in trips.columns]
    if drop_existing:
        trips.drop(columns=drop_existing, inplace=True)

    trips = trips.merge(trip_starts, on='trip_id', how='left')
    data[city]['trips'] = trips

    n_with_starts = int(trips['start_seconds'].notna().sum())
    print(f"  {city:<12}: {len(trips):>8,} trips | {n_with_starts:>8,} have valid start times")

print("\nTrip start times extracted and merged onto trips for all cities.")

  ahmedabad   :   14,323 trips |   14,323 have valid start times
  bengaluru   :   57,836 trips |   57,836 have valid start times
  davangere   :      586 trips |      586 have valid start times
  delhi       :   50,355 trips |   50,355 have valid start times
  hdbrt       :    8,669 trips |    8,669 have valid start times
  mumbai      :   99,080 trips |   99,080 have valid start times
  pune        :   10,128 trips |   10,128 have valid start times
  rajkot      :    3,192 trips |    3,192 have valid start times

Trip start times extracted and merged onto trips for all cities.


### Why the lowest `stop_sequence` defines trip start

- In GTFS, `stop_sequence` orders stops along the vehicle's trajectory (1, 2, 3... or 10, 20, 30...).
- The record with the lowest sequence number represents the **origin terminal departure**.
- Sorting by `['trip_id', 'stop_sequence']` and taking `drop_duplicates(subset=['trip_id'], keep='first')` guarantees we extract the exact departure moment for every trip without manual iterations.
- If a trip has no `stop_times` rows, its start time remains `NaN` (Rule 5: documented, not zero-filled).

In [22]:
# Phase 5, Cell 2: trips_per_hour & Wide hourly departure profile (0-23)
#
# 1. trips_per_hour: count distinct trips departing in each hour per route & day_type
# 2. Hourly wide profile: pivot table with hours 00-23 as columns
# 3. Save wide profile to processed/ folder (Rule 1)

hourly_profiles = {}   # hourly_profiles[city] = wide DataFrame

for city in CITIES:
    trips = data[city]['trips']
    if trips is None or 'start_hour' not in trips.columns:
        continue

    # Filter to trips with valid start hour
    valid_trips = trips[trips['start_hour'].notna()].copy()
    valid_trips['start_hour'] = valid_trips['start_hour'].astype(int)

    # Default day_type to 'all_days' if missing
    if 'day_type' not in valid_trips.columns:
        valid_trips['day_type'] = 'all_days'
    else:
        valid_trips['day_type'] = valid_trips['day_type'].fillna('unknown')

    # 1. Tidy groupby: trips per route, day_type, and hour
    tph = (
        valid_trips.groupby(['route_id', 'day_type', 'start_hour'])['trip_id']
        .nunique()
        .reset_index(name='trips_per_hour')
    )

    # 2. Wide table: pivot start_hour (0-23) as columns
    wide = valid_trips.pivot_table(
        index=['route_id', 'day_type'],
        columns='start_hour',
        values='trip_id',
        aggfunc='nunique',
        fill_value=0
    )

    # Ensure all 24 hours (0 to 23) exist as columns
    for h in range(24):
        if h not in wide.columns:
            wide[h] = 0

    # Reorder columns chronologically from 0 to 23
    wide = wide[list(range(24))]
    wide.columns = [f"hour_{h:02d}" for h in range(24)]
    wide = wide.reset_index()
    wide.insert(0, 'city', city)

    hourly_profiles[city] = wide

    # Save wide hourly profile (Rule 1)
    out_csv = PROCESSED_DIR / f"{city}_hourly_trip_counts.csv"
    wide.to_csv(out_csv, index=False)
    print(f"  {city:<12}: generated hourly profiles for {len(wide):>6,} route-day combinations -> {out_csv.name}")

# Display sample hourly wide rows from Pune
print("\nSample hourly wide profile (Pune, first 3 routes):")
sample_wide = hourly_profiles['pune'].head(3)
show_cols = ['route_id', 'day_type', 'hour_06', 'hour_08', 'hour_12', 'hour_18', 'hour_22']
print(sample_wide[show_cols].to_string(index=False))

  ahmedabad   : generated hourly profiles for    972 route-day combinations -> ahmedabad_hourly_trip_counts.csv
  bengaluru   : generated hourly profiles for  4,416 route-day combinations -> bengaluru_hourly_trip_counts.csv
  davangere   : generated hourly profiles for     14 route-day combinations -> davangere_hourly_trip_counts.csv
  delhi       : generated hourly profiles for  1,187 route-day combinations -> delhi_hourly_trip_counts.csv
  hdbrt       : generated hourly profiles for    630 route-day combinations -> hdbrt_hourly_trip_counts.csv
  mumbai      : generated hourly profiles for  1,115 route-day combinations -> mumbai_hourly_trip_counts.csv
  pune        : generated hourly profiles for    628 route-day combinations -> pune_hourly_trip_counts.csv
  rajkot      : generated hourly profiles for    271 route-day combinations -> rajkot_hourly_trip_counts.csv

Sample hourly wide profile (Pune, first 3 routes):
 route_id day_type  hour_06  hour_08  hour_12  hour_18  hour_22
  pune_

### Understanding the 24-hour departure profile

- **Columns `hour_00` to `hour_23`** show exact scheduled departures in each 1-hour block.
- **Peak Detection**:
  - `hour_08` – `hour_10`: Morning peak rush.
  - `hour_12` – `hour_15`: Midday base frequency.
  - `hour_17` – `hour_20`: Evening peak commute.
  - `hour_22` – `hour_04`: Night/owl service.
- Grouping by `day_type` ensures weekday rush hour spikes are not mixed with weekend schedules.

In [23]:
# Phase 5, Cell 3: Headway calculation per route and time_period
#
# Headway = time gap between consecutive trips in minutes.
# Constraints:
#   1. Must sort by start_seconds
#   2. Must group by route_id, day_type (and direction_id if present)
#   3. Routes/directions with only 1 trip get NaN (NOT 0!)
#   4. Compute median_headway_min, min, max, and median per time_period

headway_summary_list = []

for city in CITIES:
    trips = data[city]['trips']
    if trips is None or 'start_seconds' not in trips.columns:
        continue

    # 1. Slicing valid trips
    valid = trips[trips['start_seconds'].notna()].copy()
    if len(valid) == 0: continue

    # 2. Grouping columns: route_id, day_type, and direction_id (if present)
    group_cols = ['route_id', 'day_type'] if 'day_type' in valid.columns else ['route_id']
    if 'direction_id' in valid.columns:
        group_cols.append('direction_id')

    # 3. Sort strictly by start_seconds within each group
    valid = valid.sort_values(group_cols + ['start_seconds']).copy()

    # 4. Calculate consecutive time differences in minutes
    # diff() computes (trip_i - trip_{i-1}). First trip in each group gets NaN automatically!
    valid['headway_min'] = valid.groupby(group_cols)['start_seconds'].diff() / 60.0

    # 5. Overall route-level headway metrics
    route_hw = (
        valid.groupby('route_id')['headway_min']
        .agg(median_headway_min='median',
             min_headway_min='min',
             max_headway_min='max')
        .reset_index()
    )

    # 6. Median headway per time_period (morning, afternoon, evening, night)
    if 'start_time_period' in valid.columns:
        period_hw = (
            valid.groupby(['route_id', 'start_time_period'])['headway_min']
            .median()
            .unstack()
            .reset_index()
        )
        # Rename period columns
        for p in ['morning', 'afternoon', 'evening', 'night']:
            if p in period_hw.columns:
                period_hw.rename(columns={p: f"hw_{p}_median_min"}, inplace=True)
            else:
                period_hw[f"hw_{p}_median_min"] = float('nan')

        # Merge overall and period headway
        combined_hw = route_hw.merge(period_hw, on='route_id', how='left')
    else:
        combined_hw = route_hw

    # 7. Merge headway features onto the route_features dataframe created in Phase 4
    if city in route_features:
        rf = route_features[city]
        # Drop prior headway columns if re-running
        clean_cols = [c for c in combined_hw.columns if c != 'route_id' and c in rf.columns]
        if clean_cols:
            rf.drop(columns=clean_cols, inplace=True)

        rf = rf.merge(combined_hw, on='route_id', how='left')
        route_features[city] = rf

        # Save updated route features (Rule 1)
        out_hw_file = PROCESSED_DIR / f"{city}_route_features_with_headway.csv"
        rf.to_csv(out_hw_file, index=False)
        headway_summary_list.append(rf)

        n_with_hw = int(rf['median_headway_min'].notna().sum())
        n_total_r = len(rf)
        print(f"  {city:<12}: {n_total_r:>6,} routes | {n_with_hw:>6,} have calculable headway (>=2 trips) -> {out_hw_file.name}")

# Save master combined dataset across all 8 cities
if headway_summary_list:
    master_hw = pd.concat(headway_summary_list, ignore_index=True)
    master_hw_file = PROCESSED_DIR / "all_cities_route_features_with_headway.csv"
    master_hw.to_csv(master_hw_file, index=False)
    print(f"\nSaved master route features with headway to: {master_hw_file} ({len(master_hw):,} routes)")

  ahmedabad   :    978 routes |    878 have calculable headway (>=2 trips) -> ahmedabad_route_features_with_headway.csv
  bengaluru   :  4,434 routes |  2,487 have calculable headway (>=2 trips) -> bengaluru_route_features_with_headway.csv
  davangere   :     93 routes |     14 have calculable headway (>=2 trips) -> davangere_route_features_with_headway.csv
  delhi       :  1,270 routes |  1,176 have calculable headway (>=2 trips) -> delhi_route_features_with_headway.csv
  hdbrt       :    632 routes |    436 have calculable headway (>=2 trips) -> hdbrt_route_features_with_headway.csv
  mumbai      :  1,609 routes |  1,102 have calculable headway (>=2 trips) -> mumbai_route_features_with_headway.csv
  pune        :    628 routes |    617 have calculable headway (>=2 trips) -> pune_route_features_with_headway.csv
  rajkot      :    271 routes |    190 have calculable headway (>=2 trips) -> rajkot_route_features_with_headway.csv

Saved master route features with headway to: processed\all

### Why 1-trip routes get `NaN`, not `0`

- **Headway is a difference between two trips**: $\Delta t = t_2 - t_1$.
- If a route has only 1 trip in a day, there is no second trip. Mathematically, no headway interval exists.
- If we mistakenly filled this with `0`:
  - A headway of 0 minutes would imply buses arrive **simultaneously / infinitely frequently**!
  - Downstream optimization models would rank a 1-trip ghost route as the best service in the city!
- In accordance with **Rule 5**, genuine missing intervals are preserved as `NaN` and documented.

In [24]:
# Phase 5, Cell 4: Summary inspection of headway and concrete data example
#
# 1. Headway distributions per city (describe)
# 2. Real-world example comparing Trip Count, Frequency, and Headway

SEP = "=" * 95
print(SEP)
print("  PHASE 5 HEADWAY & FREQUENCY SUMMARY")
print(SEP)

for city in CITIES:
    rf = route_features.get(city)
    if rf is None or 'median_headway_min' not in rf.columns: continue

    print(f"\n{'#' * 30} {city.upper()} HEADWAY STATS {'#' * 30}")
    hw_cols = ['median_headway_min', 'min_headway_min', 'max_headway_min']
    period_cols = [c for c in ['hw_morning_median_min', 'hw_evening_median_min'] if c in rf.columns]
    print(rf[hw_cols + period_cols].describe().round(1).to_string())

# Real data example demonstrating the difference between Trip Count, Frequency, and Headway
print("\n" + "=" * 95)
print("  CONCRETE REAL DATA EXAMPLE: Trip Count vs Frequency vs Headway")
print("=" * 95)

# Select a high-frequency route in Bengaluru or Pune
example_city = 'pune'
trips_pune = data[example_city]['trips']
rf_pune    = route_features[example_city]

# Pick top route with multiple trips
top_route_id = rf_pune.sort_values(by='trip_count', ascending=False)['route_id'].iloc[0]
route_meta   = rf_pune[rf_pune['route_id'] == top_route_id].iloc[0]
route_trips  = trips_pune[trips_pune['route_id'] == top_route_id].sort_values('start_seconds')

print(f"  Route ID: {top_route_id}")
print(f"  1. DAILY TRIP COUNT       : {int(route_meta['trip_count']):,} total distinct scheduled trips across the day")
print(f"  2. OVERALL MEDIAN HEADWAY : {route_meta['median_headway_min']:.1f} minutes between consecutive buses")
print(f"  3. MORNING MEDIAN HEADWAY : {route_meta.get('hw_morning_median_min', 'N/A')} minutes (rush hour cadence)")
print(f"  4. EVENING MEDIAN HEADWAY : {route_meta.get('hw_evening_median_min', 'N/A')} minutes (evening rush cadence)")

# Show consecutive departures in morning hour
morning_trips = route_trips[route_trips['start_hour'] == 8].head(5)
if len(morning_trips) > 1:
    print("\n  Sample consecutive departures in Hour 08:00 - 08:59:")
    for idx, (_, r) in enumerate(morning_trips.iterrows()):
        m = int(r['start_minute'])
        s = int(r['start_seconds'])
        print(f"    Departure {idx+1}: 08:{m:02d}:00  ({s} seconds from midnight)")
    print(f"  --> Frequency during Hour 8 = {len(morning_trips)} departures / hour")
print("=" * 95)

  PHASE 5 HEADWAY & FREQUENCY SUMMARY

############################## AHMEDABAD HEADWAY STATS ##############################
       median_headway_min  min_headway_min  max_headway_min  hw_morning_median_min  hw_evening_median_min
count               878.0            878.0            878.0                  621.0                  616.0
mean                 72.1             56.4            175.7                   56.6                   93.0
std                  86.1             83.7            139.4                   51.0                  102.3
min                   5.0              0.0              5.0                    1.0                    5.0
25%                  25.0             15.0             55.0                   20.0                   27.5
50%                  40.0             30.0            135.0                   40.0                   50.0
75%                  77.5             60.0            275.0                   70.0                  125.0
max                 571.0  

### Summary: Trip Count vs. Frequency vs. Headway

| Concept | Definition | Unit | Why It Matters |
|---|---|:---:|---|
| **Trip Count** | Total number of trips scheduled across the day | `trips/day` | Measures total capacity and resource allocation |
| **Frequency** | Number of trips departing within a specific hour | `trips/hour` | Captures hourly service intensity across peak/off-peak |
| **Headway** | Time gap between consecutive departures on the same route | `minutes` | Directly measures passenger waiting time and service predictability |

---
**Phase 5 complete.**

---
# Phase 6: Estimate Route Length per Route

**Goal:** Estimate the physical trajectory length (in kilometers) for every route in all 8 cities.

### Why Straight-Line Distance is NOT Road Distance:
1. **Road network geometry**: Vehicles do not travel as the crow flies. Roads curve around topography, water bodies, railway crossings, and dense urban street grids.
2. **Stop-to-stop chord error**: A straight line connecting stop A to stop B ignores road bends between them. Summing straight-line stop connections always **underestimates** actual road travel distance (often by 15% to 35%).
3. **High-resolution shapes**: When `shapes.txt` is available, it provides fine-grained GPS coordinate vertices tracing the exact road path.

### Why Number of Stops is NOT Route Length:
- **Stop density varies dramatically**: A 10 km route in a crowded old city core might have **45 stops** spaced 220 m apart.
- An express, airport shuttle, or suburban highway route of the exact same 10 km distance might have only **3 stops** spaced 3.3 km apart.
- Stop count reflects **passenger accessibility and boarding points**, while route length measures **spatial operational extent**.

### Two Methods Applied in this Phase:
- **Method 1 (`shape`)**: Used when `shapes.txt` exists and trips have `shape_id`. We sort shape points by `shape_pt_sequence` and compute consecutive Haversine distances for the route's representative shape (most frequent `shape_id` among trips).
- **Method 2 (`stop_straight_line_estimate`)**: Fallback when shapes are absent. We pick the representative trip with the most stops, sort stops by `stop_sequence`, and sum Haversine distances between consecutive stop coordinates.

> **Rule 5 Compliance**: Missing coordinates yield `NaN`, **never `0`**. Routes under 0.5 km or over 150 km are flagged for review without removing any data.

In [25]:
# Phase 6, Cell 1: Haversine distance functions
#
# Haversine formula computes great-circle distance between coordinates on Earth.
# Earth mean radius R = 6,371.0088 km.

import numpy as np

def haversine_distance_km(lat1, lon1, lat2, lon2):
    """
    Calculate the great circle distance in kilometers between two points
    (or two equal-length arrays of points) on Earth.
    """
    R = 6371.0088  # Earth mean radius in kilometers
    
    phi1 = np.radians(lat1)
    phi2 = np.radians(lat2)
    delta_phi = np.radians(lat2 - lat1)
    delta_lambda = np.radians(lon2 - lon1)
    
    a = np.sin(delta_phi / 2.0)**2 + np.cos(phi1) * np.cos(phi2) * np.sin(delta_lambda / 2.0)**2
    c = 2.0 * np.arcsin(np.clip(np.sqrt(a), 0.0, 1.0))
    return R * c


def calculate_path_length_km(df, lat_col, lon_col):
    """
    Sum Haversine distances between consecutive coordinate pairs in an ordered DataFrame.
    Returns NaN if df has fewer than 2 points or contains NaN coordinates.
    """
    if df is None or len(df) < 2:
        return float('nan')
    
    lats = pd.to_numeric(df[lat_col], errors='coerce').values
    lons = pd.to_numeric(df[lon_col], errors='coerce').values
    
    # If any coordinates are NaN, distance cannot be accurately determined
    if np.isnan(lats).any() or np.isnan(lons).any():
        return float('nan')
    
    # Consecutive step distances
    step_dists = haversine_distance_km(lats[:-1], lons[:-1], lats[1:], lons[1:])
    return float(np.sum(step_dists))


# Quick sanity test: Delhi to Mumbai (~1150 km straight-line)
test_dist = haversine_distance_km(28.6139, 77.2090, 19.0760, 72.8777)
print(f"Haversine test -- New Delhi to Mumbai: {test_dist:.2f} km (expected ~1,148 - 1,152 km)")
assert 1140 < test_dist < 1160, "Haversine formula test failed!"
print("Haversine calculation verified successfully.")

Haversine test -- New Delhi to Mumbai: 1148.10 km (expected ~1,148 - 1,152 km)
Haversine calculation verified successfully.


In [26]:
# Phase 6, Cell 2: Estimate route length per route for all 8 cities
#
# Method 1 (Shapes): Used when shapes.txt exists and trips has shape_id.
#   - Finds representative shape_id (mode among trips)
#   - Sorts by shape_pt_sequence, sums Haversine distances
#   - Checks shape_dist_traveled unit if present
#
# Method 2 (Stops Fallback): Used when shapes are unavailable.
#   - Picks representative trip (trip with most stops)
#   - Sorts stops by stop_sequence, sums Haversine distances across consecutive stops
#   - Marks as 'stop_straight_line_estimate'
#
# Rule 5: Missing coordinates = NaN, never 0.

length_feature_list = []
shape_dist_unit_report = []

for city in CITIES:
    city_path = BASE_DIR / CITIES[city]
    ext = ".csv" if city in CSV_CITIES else ".txt"
    shape_file = city_path / f"shapes{ext}"
    
    trips = data[city]['trips']
    routes = data[city]['routes']
    stops = data[city]['stops']
    st = data[city]['stop_times']
    
    if routes is None or trips is None:
        continue
    
    # Check if shapes file exists AND trips has shape_id with non-null values
    has_shapes = shape_file.exists() and ('shape_id' in trips.columns) and (trips['shape_id'].notna().sum() > 0)
    
    route_length_map = {}
    length_method_map = {}
    
    # -------------------------------------------------------------------------
    # METHOD 1: SHAPES
    # -------------------------------------------------------------------------
    if has_shapes:
        try:
            # Read shapes file with numeric dtypes
            raw_shapes = pd.read_csv(shape_file, low_memory=False)
            # Prefix shape_id to match trips['shape_id']
            raw_shapes['shape_id'] = city + "_" + raw_shapes['shape_id'].astype(str)
            raw_shapes['shape_pt_sequence'] = pd.to_numeric(raw_shapes['shape_pt_sequence'], errors='coerce')
            raw_shapes['shape_pt_lat'] = pd.to_numeric(raw_shapes['shape_pt_lat'], errors='coerce')
            raw_shapes['shape_pt_lon'] = pd.to_numeric(raw_shapes['shape_pt_lon'], errors='coerce')
            
            # Check shape_dist_traveled if present and verify unit
            if 'shape_dist_traveled' in raw_shapes.columns:
                raw_shapes['shape_dist_traveled'] = pd.to_numeric(raw_shapes['shape_dist_traveled'], errors='coerce')
                sample_sid = raw_shapes['shape_id'].iloc[0]
                sample_pts = raw_shapes[raw_shapes['shape_id'] == sample_sid].sort_values('shape_pt_sequence')
                calc_h_km = calculate_path_length_km(sample_pts, 'shape_pt_lat', 'shape_pt_lon')
                max_sdt = sample_pts['shape_dist_traveled'].max()
                
                if pd.notna(calc_h_km) and calc_h_km > 0 and pd.notna(max_sdt):
                    ratio = max_sdt / calc_h_km
                    unit_guess = "meters (m)" if 800 < ratio < 1200 else ("kilometers (km)" if 0.8 < ratio < 1.2 else "miles / other")
                    shape_dist_unit_report.append({
                        'city': city,
                        'sample_shape_id': sample_sid,
                        'haversine_km': round(calc_h_km, 2),
                        'max_shape_dist_traveled': round(max_sdt, 2),
                        'ratio_sdt_to_haversine': round(ratio, 2),
                        'inferred_unit': unit_guess
                    })
            
            # Precompute length for each unique shape_id in one grouped pass
            shape_lengths = {}
            for sid, grp in raw_shapes.groupby('shape_id'):
                sorted_grp = grp.sort_values('shape_pt_sequence')
                shape_lengths[sid] = calculate_path_length_km(sorted_grp, 'shape_pt_lat', 'shape_pt_lon')
            
            # For each route, pick most frequent shape_id among trips (mode)
            valid_trip_shapes = trips[trips['shape_id'].notna() & (trips['shape_id'] != '') & (trips['shape_id'] != f'{city}_nan')]
            for r_id, grp in valid_trip_shapes.groupby('route_id'):
                modes = grp['shape_id'].mode()
                if not modes.empty:
                    rep_shape = modes.iloc[0]
                    if rep_shape in shape_lengths and pd.notna(shape_lengths[rep_shape]):
                        route_length_map[r_id] = shape_lengths[rep_shape]
                        length_method_map[r_id] = "shape"
        except Exception as e:
            print(f"  {city}: Error reading shapes ({e}), falling back to stops.")
    
    # -------------------------------------------------------------------------
    # METHOD 2: STOPS FALLBACK (for cities without shapes OR routes missing shapes)
    # -------------------------------------------------------------------------
    if st is not None and stops is not None and 'stop_lat' in stops.columns and 'stop_lon' in stops.columns:
        stops_coords = stops[['stop_id', 'stop_lat', 'stop_lon']].copy()
        stops_coords['stop_lat'] = pd.to_numeric(stops_coords['stop_lat'], errors='coerce')
        stops_coords['stop_lon'] = pd.to_numeric(stops_coords['stop_lon'], errors='coerce')
        
        st_clean = st[['trip_id', 'stop_id', 'stop_sequence']].copy()
        st_clean['stop_sequence'] = pd.to_numeric(st_clean['stop_sequence'], errors='coerce')
        st_with_coords = st_clean.merge(stops_coords, on='stop_id', how='left')
        
        # Count valid coordinate stops per trip
        trip_stop_counts = st_with_coords.dropna(subset=['stop_lat', 'stop_lon']).groupby('trip_id')['stop_sequence'].count()
        
        # Fill unmeasured routes
        routes_to_fill = [r for r in routes['route_id'].unique() if r not in route_length_map]
        trips_to_fill = trips[trips['route_id'].isin(routes_to_fill)]
        
        for r_id, grp in trips_to_fill.groupby('route_id'):
            route_trip_ids = grp['trip_id'].values
            avail_counts = trip_stop_counts.reindex(route_trip_ids).dropna()
            if not avail_counts.empty:
                rep_trip_id = avail_counts.idxmax()  # representative trip with most stops
                rep_stops = st_with_coords[st_with_coords['trip_id'] == rep_trip_id].sort_values('stop_sequence')
                calc_len = calculate_path_length_km(rep_stops, 'stop_lat', 'stop_lon')
                route_length_map[r_id] = calc_len if pd.notna(calc_len) else float('nan')
                length_method_map[r_id] = "stop_straight_line_estimate"
            else:
                route_length_map[r_id] = float('nan')
                length_method_map[r_id] = "stop_straight_line_estimate"
    
    # -------------------------------------------------------------------------
    # Merge into route_features[city]
    # -------------------------------------------------------------------------
    if city in route_features:
        rf = route_features[city]
        
        rf['route_length_km'] = rf['route_id'].map(route_length_map)
        rf['length_method']    = rf['route_id'].map(length_method_map).fillna('unmeasured')
        
        # Flags: < 0.5 km or > 150 km (Rule 5: flag only, do not remove!)
        rf['flag_too_short']     = rf['route_length_km'] < 0.5
        rf['flag_too_long']      = rf['route_length_km'] > 150.0
        rf['flag_length_review'] = rf['flag_too_short'] | rf['flag_too_long']
        
        route_features[city] = rf
        
        # Save updated route features (Rule 1)
        out_rf_file = PROCESSED_DIR / f"{city}_route_features_with_length.csv"
        rf.to_csv(out_rf_file, index=False)
        length_feature_list.append(rf)
        
        n_shapes = int((rf['length_method'] == 'shape').sum())
        n_stops  = int((rf['length_method'] == 'stop_straight_line_estimate').sum())
        n_flags  = int(rf['flag_length_review'].sum())
        print(f"  {city:<12}: {len(rf):>6,} routes | shapes={n_shapes:>5,} | stop_estimates={n_stops:>5,} | flagged={n_flags:>4,} -> {out_rf_file.name}")

# Combined master file across all cities
if length_feature_list:
    master_len_df = pd.concat(length_feature_list, ignore_index=True)
    master_len_file = PROCESSED_DIR / "all_cities_route_features_with_length.csv"
    master_len_df.to_csv(master_len_file, index=False)
    print(f"\nSaved master route features with length to: {master_len_file} ({len(master_len_df):,} total routes)")

  ahmedabad   :    978 routes | shapes=    0 | stop_estimates=  972 | flagged=   6 -> ahmedabad_route_features_with_length.csv
  bengaluru   :  4,434 routes | shapes=4,416 | stop_estimates=    0 | flagged=  84 -> bengaluru_route_features_with_length.csv
  davangere   :     93 routes | shapes=    0 | stop_estimates=   14 | flagged=   0 -> davangere_route_features_with_length.csv
  delhi       :  1,270 routes | shapes=    0 | stop_estimates=1,187 | flagged=   0 -> delhi_route_features_with_length.csv
  hdbrt       :    632 routes | shapes=    0 | stop_estimates=  630 | flagged=   3 -> hdbrt_route_features_with_length.csv
  mumbai      :  1,609 routes | shapes=    0 | stop_estimates=1,115 | flagged=  14 -> mumbai_route_features_with_length.csv
  pune        :    628 routes | shapes=  628 | stop_estimates=    0 | flagged=   0 -> pune_route_features_with_length.csv
  rajkot      :    271 routes | shapes=    0 | stop_estimates=  271 | flagged=   0 -> rajkot_route_features_with_length.csv

Sa

### Step-by-step breakdown of route length estimation

1. **Representative Shape Selection (Mode)**:
   - A single route may have variations (e.g. slight terminus extensions or school trip detours).
   - We take the mode (`grp['shape_id'].mode().iloc[0]`) to select the predominant main-line shape.
2. **`shape_dist_traveled` Unit Detection**:
   - When `shape_dist_traveled` exists (such as in Bengaluru), we compare it against our Haversine calculation.
   - A ratio of ~1,000 indicates the GTFS field was recorded in **meters**, while ~1.0 indicates **kilometers**.
3. **Representative Trip Selection (Max Stops Fallback)**:
   - In cities without `shapes.txt`, we pick the trip with the greatest count of valid coordinates.
   - This avoids short-turning depot trips and captures the full route extent.
4. **Flagging for Review**:
   - Routes `< 0.5 km`: often circular loop artifacts, depot moves, or truncated feeds.
   - Routes `> 150 km`: regional state-transport inter-city services or coordinate typos.
   - Flagging preserves data transparency without destructive dropping.

In [27]:
# Phase 6, Cell 3: Method counts, summary statistics, and flagged routes
#
# 1. shape_dist_traveled unit report
# 2. Length estimation method counts per city
# 3. route_length_km summary statistics per city
# 4. Sample flagged routes

SEP = "=" * 95
print(SEP)
print("  PHASE 6 ROUTE LENGTH ESTIMATION SUMMARY")
print(SEP)

# 1. shape_dist_traveled unit analysis
if shape_dist_unit_report:
    print("\n--- A. shape_dist_traveled Unit Comparison ---")
    sdt_df = pd.DataFrame(shape_dist_unit_report)
    print(sdt_df.to_string(index=False))

# 2. Method counts per city
print("\n--- B. Length Estimation Method Counts per City ---")
print(f"{'City':<12} {'shape':>10} {'stop_estimate':>16} {'unmeasured/NaN':>16}")
print("-" * 58)
for city in CITIES:
    rf = route_features.get(city)
    if rf is None or 'length_method' not in rf.columns: continue
    vc = rf['length_method'].value_counts()
    n_shape = vc.get('shape', 0)
    n_stop  = vc.get('stop_straight_line_estimate', 0)
    n_unm   = vc.get('unmeasured', 0) + int(rf['route_length_km'].isna().sum())
    print(f"{city:<12} {n_shape:>10,} {n_stop:>16,} {n_unm:>16,}")

# 3. Summary statistics per city
print("\n--- C. route_length_km Statistics (describe) per City ---")
for city in CITIES:
    rf = route_features.get(city)
    if rf is None or 'route_length_km' not in rf.columns: continue
    desc = rf['route_length_km'].describe().round(2)
    print(f"\n  {city.upper()}: count={int(desc['count']):,}, mean={desc['mean']} km, std={desc['std']} km, "
          f"min={desc['min']} km, 25%={desc['25%']} km, median={desc['50%']} km, 75%={desc['75%']} km, max={desc['max']} km")

# 4. Flagged routes (< 0.5 km or > 150 km)
print("\n--- D. Sample Routes Flagged for Review (< 0.5 km or > 150 km) ---")
total_flagged = 0
for city in CITIES:
    rf = route_features.get(city)
    if rf is None or 'flag_length_review' not in rf.columns: continue
    flagged = rf[rf['flag_length_review'] == True]
    total_flagged += len(flagged)
    if len(flagged) > 0:
        disp = ['route_id', 'route_length_km', 'length_method', 'flag_too_short', 'flag_too_long']
        print(f"\n  {city.upper()} ({len(flagged)} flagged routes total):")
        print(flagged[disp].head(4).to_string(index=False))

print(f"\nTotal routes flagged across all cities: {total_flagged:,}")
print("\n" + SEP)

  PHASE 6 ROUTE LENGTH ESTIMATION SUMMARY

--- A. shape_dist_traveled Unit Comparison ---
     city    sample_shape_id  haversine_km  max_shape_dist_traveled  ratio_sdt_to_haversine inferred_unit
bengaluru bengaluru_104 DOWN          4.71                  4714.54                 1001.42    meters (m)

--- B. Length Estimation Method Counts per City ---
City              shape    stop_estimate   unmeasured/NaN
----------------------------------------------------------
ahmedabad             0              972               12
bengaluru         4,416                0               36
davangere             0               14              158
delhi                 0            1,187              166
hdbrt                 0              630                4
mumbai                0            1,115              988
pune                628                0                0
rajkot                0              271                0

--- C. route_length_km Statistics (describe) per City ---

  AH

### Phase 6 Summary & Method Comparison

| Method | When Used | Precision | Caveat |
|---|---|:---:|---|
| **`shape`** | When `shapes.txt` is provided (Bengaluru, Pune) | **High** | Follows physical road polylines with fine GPS curvature |
| **`stop_straight_line_estimate`** | When shapes are absent (fallback for 6 cities) | **Moderate** | Great-circle chord between stops; systematically **underestimates** true road distance by ~15–35% |

> **Data Integrity (Rule 5)**: Routes with missing stop GPS coordinates remain `NaN`. Flagged outlier routes (< 0.5 km, > 150 km) are preserved for mentor review rather than arbitrarily deleted.

---
**Phase 6 complete.**

---
# Phase 7: Final Route Profile Assembly & Persistence

**Goal:** Consolidate all route-level features engineered across Phases 1 to 6 into a single, standardized master table: `processed/route_features.csv`.

### Schema Specification (20 Columns):
1. **Core Identifiers & Names**: `city`, `route_id` (prefixed), `original_route_id` (raw), `route_short_name`, `route_long_name`
2. **Network Geometry & Scale**: `trip_count`, `stop_count`, `route_length_km`, `length_method`
3. **Cadence & Headway**: `median_headway_min`
4. **Diurnal Time-Period Distribution**: `trips_morning`, `trips_afternoon`, `trips_evening`, `trips_night`
5. **Weekly Schedule Profile**: `trips_weekday`, `trips_weekend`, `operating_days`
6. **Operational Window**: `first_departure_hour`, `last_departure_hour`, `service_span_hours`

### Excluded Candidate Features (Rule 2 & Data Integrity):
- **Passenger counts / ridership / overcrowding**: Strictly excluded per **Rule 4** (GTFS does not contain ridership data).
- **Fares**: Available only for 2 of 8 cities; excluded to maintain cross-city consistency.
- **Live travel speeds / delays**: GTFS is a scheduled specification, not real-time AVL/telemetry.
- **Circuity / curvature**: Shapes only exist for Bengaluru and Pune, so circuity cannot be reliably derived across all cities.

In [28]:
# Phase 7, Cell 1: Assemble final route features table across all 8 cities
#
# 1. Extract and standardize all 20 specified columns
# 2. Preserve identifier string types
# 3. Save to processed/route_features.csv

from pathlib import Path
import numpy as np
import pandas as pd

PROCESSED_DIR = Path("processed")
PROCESSED_DIR.mkdir(exist_ok=True)

WEEKDAYS = {'monday', 'tuesday', 'wednesday', 'thursday', 'friday'}
WEEKENDS = {'saturday', 'sunday'}

def has_weekday(days_str):
    if not isinstance(days_str, str): return False
    days = set(days_str.lower().split(','))
    return len(days & WEEKDAYS) > 0

def has_weekend(days_str):
    if not isinstance(days_str, str): return False
    days = set(days_str.lower().split(','))
    return len(days & WEEKENDS) > 0

def union_days(series):
    active_days = set()
    for s in series.dropna():
        if isinstance(s, str) and s not in ['unknown', 'none']:
            active_days.update(s.lower().split(','))
    DAY_ORDER = ['monday', 'tuesday', 'wednesday', 'thursday', 'friday', 'saturday', 'sunday']
    ordered = [d for d in DAY_ORDER if d in active_days]
    return ','.join(ordered) if ordered else 'unknown'

master_profiles = []

for city in CITIES:
    if city not in route_features:
        continue

    rf = route_features[city].copy()
    trips = data[city]['trips']

    # -------------------------------------------------------------------------
    # 1. Identifiers and basic fields
    # -------------------------------------------------------------------------
    prefix = city + "_"
    rf['original_route_id'] = rf['route_id'].apply(
        lambda x: str(x)[len(prefix):] if str(x).startswith(prefix) else str(x)
    )
    rf['route_id'] = rf['route_id'].astype(str)
    rf['original_route_id'] = rf['original_route_id'].astype(str)
    rf['city'] = city

    if 'route_short_name' not in rf.columns: rf['route_short_name'] = float('nan')
    if 'route_long_name' not in rf.columns:  rf['route_long_name']  = float('nan')

    # -------------------------------------------------------------------------
    # 2. Trip breakdowns (time_period, weekdays, weekends, span)
    # -------------------------------------------------------------------------
    if trips is not None and not trips.empty:
        trips_valid = trips.copy()

        # A. Trips per time_period (morning, afternoon, evening, night)
        if 'start_time_period' in trips_valid.columns:
            tp_pivot = (
                trips_valid.dropna(subset=['start_time_period'])
                .groupby(['route_id', 'start_time_period'])['trip_id']
                .nunique()
                .unstack(fill_value=0)
                .reset_index()
            )
            for p in ['morning', 'afternoon', 'evening', 'night']:
                if p not in tp_pivot.columns: tp_pivot[p] = 0
            tp_pivot.rename(columns={
                'morning': 'trips_morning',
                'afternoon': 'trips_afternoon',
                'evening': 'trips_evening',
                'night': 'trips_night'
            }, inplace=True)
        else:
            tp_pivot = pd.DataFrame(columns=['route_id', 'trips_morning', 'trips_afternoon', 'trips_evening', 'trips_night'])

        # B. Weekday vs Weekend trips
        if 'operating_days' in trips_valid.columns:
            trips_valid['is_wd'] = trips_valid['operating_days'].apply(has_weekday)
            trips_valid['is_we'] = trips_valid['operating_days'].apply(has_weekend)

            wd_counts = trips_valid[trips_valid['is_wd']].groupby('route_id')['trip_id'].nunique().reset_index(name='trips_weekday')
            we_counts = trips_valid[trips_valid['is_we']].groupby('route_id')['trip_id'].nunique().reset_index(name='trips_weekend')
        else:
            wd_counts = pd.DataFrame(columns=['route_id', 'trips_weekday'])
            we_counts = pd.DataFrame(columns=['route_id', 'trips_weekend'])

        # C. First departure, last departure, and service span
        if 'start_seconds' in trips_valid.columns:
            span_stats = trips_valid.dropna(subset=['start_seconds']).groupby('route_id')['start_seconds'].agg(min_s='min', max_s='max').reset_index()
            span_stats['first_departure_hour'] = span_stats['min_s'] // 3600
            span_stats['last_departure_hour']  = span_stats['max_s'] // 3600
            span_stats['service_span_hours']   = ((span_stats['max_s'] - span_stats['min_s']) / 3600.0).round(2)
        else:
            span_stats = pd.DataFrame(columns=['route_id', 'first_departure_hour', 'last_departure_hour', 'service_span_hours'])

        # D. Route-level union of operating days
        if 'operating_days' in trips_valid.columns:
            route_days = trips_valid.groupby('route_id')['operating_days'].apply(union_days).reset_index(name='operating_days')
        else:
            route_days = pd.DataFrame(columns=['route_id', 'operating_days'])

    else:
        tp_pivot   = pd.DataFrame(columns=['route_id', 'trips_morning', 'trips_afternoon', 'trips_evening', 'trips_night'])
        wd_counts  = pd.DataFrame(columns=['route_id', 'trips_weekday'])
        we_counts  = pd.DataFrame(columns=['route_id', 'trips_weekend'])
        span_stats = pd.DataFrame(columns=['route_id', 'first_departure_hour', 'last_departure_hour', 'service_span_hours'])
        route_days = pd.DataFrame(columns=['route_id', 'operating_days'])

    # -------------------------------------------------------------------------
    # 3. Clean left joins into route profile
    # -------------------------------------------------------------------------
    for sub_df in [tp_pivot, wd_counts, we_counts, span_stats[['route_id', 'first_departure_hour', 'last_departure_hour', 'service_span_hours']], route_days]:
        # Drop overlapping columns if already present in rf
        overlap = [c for c in sub_df.columns if c != 'route_id' and c in rf.columns]
        if overlap:
            rf.drop(columns=overlap, inplace=True)
        rf = rf.merge(sub_df, on='route_id', how='left')

    # Handle routes with trips: fill counts with 0; if route has NO trips, leave NaN
    has_trips = rf['trip_count'].notna() & (rf['trip_count'] > 0)
    count_cols = ['trips_morning', 'trips_afternoon', 'trips_evening', 'trips_night', 'trips_weekday', 'trips_weekend']
    for c in count_cols:
        if c in rf.columns:
            rf.loc[has_trips, c] = rf.loc[has_trips, c].fillna(0)

    rf['operating_days'] = rf['operating_days'].fillna('unknown')
    master_profiles.append(rf)

# -----------------------------------------------------------------------------
# 4. Combine all cities and enforce strict 20-column schema
# -----------------------------------------------------------------------------
FINAL_COLS = [
    'city',
    'route_id',
    'original_route_id',
    'route_short_name',
    'route_long_name',
    'trip_count',
    'stop_count',
    'route_length_km',
    'length_method',
    'median_headway_min',
    'trips_morning',
    'trips_afternoon',
    'trips_evening',
    'trips_night',
    'trips_weekday',
    'trips_weekend',
    'first_departure_hour',
    'last_departure_hour',
    'service_span_hours',
    'operating_days'
]

all_cities_profile = pd.concat(master_profiles, ignore_index=True)

# Ensure all columns exist
for col in FINAL_COLS:
    if col not in all_cities_profile.columns:
        all_cities_profile[col] = float('nan')

final_route_features = all_cities_profile[FINAL_COLS].copy()

# Ensure string identifier types
final_route_features['city'] = final_route_features['city'].astype(str)
final_route_features['route_id'] = final_route_features['route_id'].astype(str)
final_route_features['original_route_id'] = final_route_features['original_route_id'].astype(str)

# Save final dataset (Rule 1)
output_file = PROCESSED_DIR / "route_features.csv"
final_route_features.to_csv(output_file, index=False)

print("=" * 85)
print(f"  FINAL ROUTE FEATURES SAVED SUCCESSFULLY -> {output_file}")
print("=" * 85)
print(f"  Total Rows (Routes) : {final_route_features.shape[0]:,}")
print(f"  Total Columns       : {final_route_features.shape[1]}")
print(f"  Cities Covered      : {list(final_route_features['city'].unique())}")
print("\nFirst 3 rows (selected columns):")
sample_cols = ['city', 'route_id', 'trip_count', 'stop_count', 'route_length_km', 'median_headway_min', 'service_span_hours']
print(final_route_features[sample_cols].head(3).to_string(index=False))

  FINAL ROUTE FEATURES SAVED SUCCESSFULLY -> processed\route_features.csv
  Total Rows (Routes) : 9,915
  Total Columns       : 20
  Cities Covered      : ['ahmedabad', 'bengaluru', 'davangere', 'delhi', 'hdbrt', 'mumbai', 'pune', 'rajkot']

First 3 rows (selected columns):
     city           route_id  trip_count  stop_count  route_length_km  median_headway_min  service_span_hours
ahmedabad ahmedabad_AMC_1790         5.0        44.0        44.528570               162.5               11.67
ahmedabad ahmedabad_AMC_1791         5.0        44.0        44.528570               160.0               10.92
ahmedabad ahmedabad_AMC_1798        14.0        71.0        22.365598                55.0               13.08


### How Phase 7 Features Were Constructed

1. **`original_route_id`**:
   - Restored by dynamically stripping the `"{city}_"` prefix from `route_id`.
   - Keeps global uniqueness via `route_id` while allowing traceability back to raw agency feeds.
2. **Time-Period Breakdowns (`trips_morning`, `trips_afternoon`, `trips_evening`, `trips_night`)**:
   - Distinct trips departing in each respective time window.
   - If a route runs trips, empty periods are filled with `0` (indicating zero scheduled departures in that window). Routes with no trips retain `NaN`.
3. **Weekly Operating Profiles (`trips_weekday`, `trips_weekend`, `operating_days`)**:
   - Derived from `calendar` service specifications joined to trips.
   - `operating_days` computes the unified set of active days across all trips on that route.
4. **Service Span (`service_span_hours`)**:
   - Calculated as $(t_{\text{last}} - t_{\text{first}}) / 3600.0$ using `start_seconds`.
   - Accurately measures route operating hours without midnight-wrap errors (overnight trips $\ge 24\text{h}$ are natively supported).

In [29]:
# Phase 7, Cell 2: Feature categorization, dtypes, and missing value audit
#
# 1. Feature taxonomy table: Numerical vs Categorical, Original vs Derived
# 2. Column dtypes
# 3. Missing value count and percentage per column

SEP = "=" * 95
print(SEP)
print("  PHASE 7 DATASET AUDIT: Feature Taxonomy, Dtypes & Missing Values")
print(SEP)

# 1. Feature taxonomy table
taxonomy = pd.DataFrame([
    {"column": "city",                 "category": "Categorical", "origin": "Original GTFS (Folder)", "dtype": "str",     "description": "City transit system identifier"},
    {"column": "route_id",             "category": "Identifier",  "origin": "Derived (Phase 1)",        "dtype": "str",     "description": "Globally unique prefixed route identifier"},
    {"column": "original_route_id",    "category": "Identifier",  "origin": "Original GTFS (routes.txt)","dtype": "str",   "description": "Raw route identifier as published by agency"},
    {"column": "route_short_name",     "category": "Categorical", "origin": "Original GTFS (routes.txt)","dtype": "str",   "description": "Public route designation / line number"},
    {"column": "route_long_name",      "category": "Categorical", "origin": "Original GTFS (routes.txt)","dtype": "str",   "description": "Full route origin and terminus description"},
    {"column": "trip_count",           "category": "Numerical",   "origin": "Derived (Phase 4)",         "dtype": "float64", "description": "Total distinct scheduled trips per day"},
    {"column": "stop_count",           "category": "Numerical",   "origin": "Derived (Phase 4)",         "dtype": "float64", "description": "Distinct physical stops served by route"},
    {"column": "route_length_km",      "category": "Numerical",   "origin": "Derived (Phase 6)",         "dtype": "float64", "description": "Estimated route path length in kilometers"},
    {"column": "length_method",        "category": "Categorical", "origin": "Derived (Phase 6)",         "dtype": "str",     "description": "Measurement method: shape or stop estimate"},
    {"column": "median_headway_min",   "category": "Numerical",   "origin": "Derived (Phase 5)",         "dtype": "float64", "description": "Median departure headway in minutes"},
    {"column": "trips_morning",        "category": "Numerical",   "origin": "Derived (Phase 5/7)",       "dtype": "float64", "description": "Trips departing in morning (05:00-11:59)"},
    {"column": "trips_afternoon",      "category": "Numerical",   "origin": "Derived (Phase 5/7)",       "dtype": "float64", "description": "Trips departing in afternoon (12:00-16:59)"},
    {"column": "trips_evening",        "category": "Numerical",   "origin": "Derived (Phase 5/7)",       "dtype": "float64", "description": "Trips departing in evening (17:00-20:59)"},
    {"column": "trips_night",          "category": "Numerical",   "origin": "Derived (Phase 5/7)",       "dtype": "float64", "description": "Trips departing in night (21:00-04:59)"},
    {"column": "trips_weekday",        "category": "Numerical",   "origin": "Derived (Phase 3/7)",       "dtype": "float64", "description": "Trips active on Monday through Friday"},
    {"column": "trips_weekend",        "category": "Numerical",   "origin": "Derived (Phase 3/7)",       "dtype": "float64", "description": "Trips active on Saturday or Sunday"},
    {"column": "first_departure_hour", "category": "Numerical",   "origin": "Derived (Phase 5/7)",       "dtype": "float64", "description": "Hour of first scheduled daily departure"},
    {"column": "last_departure_hour",  "category": "Numerical",   "origin": "Derived (Phase 5/7)",       "dtype": "float64", "description": "Hour of last scheduled daily departure"},
    {"column": "service_span_hours",   "category": "Numerical",   "origin": "Derived (Phase 5/7)",       "dtype": "float64", "description": "Total daily operational span in hours"},
    {"column": "operating_days",       "category": "Categorical", "origin": "Derived (Phase 3/7)",       "dtype": "str",     "description": "Union of active days of week for route"}
])

print("\n--- A. Feature Taxonomy Table ---")
print(taxonomy.to_string(index=False))

# 2. Summary counts by taxonomy
print("\nFeature Breakdown:")
print(f"  Numerical features   : {(taxonomy['category'] == 'Numerical').sum()}")
print(f"  Categorical features : {(taxonomy['category'] == 'Categorical').sum()}")
print(f"  Identifier fields    : {(taxonomy['category'] == 'Identifier').sum()}")
print(f"  Original GTFS fields : {(taxonomy['origin'].str.startswith('Original')).sum()}")
print(f"  Derived features     : {(taxonomy['origin'].str.startswith('Derived')).sum()}")

# 3. Missing value audit
print("\n--- B. Missing Values and Data Types in processed/route_features.csv ---")
null_counts = final_route_features.isnull().sum()
null_pcts   = (100.0 * null_counts / len(final_route_features)).round(2)
dtypes      = final_route_features.dtypes

audit_df = pd.DataFrame({
    'dtype': dtypes,
    'missing_count': null_counts,
    'missing_pct': null_pcts
}).reset_index().rename(columns={'index': 'column'})

print(audit_df.to_string(index=False))

# 4. Verification of Rule 4 (Zero passenger columns)
passenger_keywords = ['passenger', 'ridership', 'crowd', 'load', 'occupancy']
found_prohibited = [c for c in final_route_features.columns if any(k in c.lower() for k in passenger_keywords)]
print(f"\nRule 4 Check -- Prohibited passenger/crowding columns found: {found_prohibited} (OK: empty list)")
assert len(found_prohibited) == 0, "Rule 4 violation detected!"
print("=" * 95)

  PHASE 7 DATASET AUDIT: Feature Taxonomy, Dtypes & Missing Values

--- A. Feature Taxonomy Table ---
              column    category                     origin   dtype                                 description
                city Categorical     Original GTFS (Folder)     str              City transit system identifier
            route_id  Identifier          Derived (Phase 1)     str   Globally unique prefixed route identifier
   original_route_id  Identifier Original GTFS (routes.txt)     str Raw route identifier as published by agency
    route_short_name Categorical Original GTFS (routes.txt)     str      Public route designation / line number
     route_long_name Categorical Original GTFS (routes.txt)     str  Full route origin and terminus description
          trip_count   Numerical          Derived (Phase 4) float64      Total distinct scheduled trips per day
          stop_count   Numerical          Derived (Phase 4) float64     Distinct physical stops served by route
  

### Module 2 Complete — Final Summary

All objectives for **Module 2: Feature Engineering** are now complete.

#### Summary of Deliverables:
- **`processed/route_features.csv`**: Master consolidated dataset (one row per route) containing 20 rigorously validated features across all 8 Indian cities.
- **Strict Rule Adherence**:
  - Raw GTFS files unmodified (Rule 1).
  - GTFS hours $\ge 24\text{h}$ handled with integer second arithmetic (Rule 2).
  - `nunique()` strictly used for trip counts, preventing stop-times row inflation (Rule 3).
  - Zero passenger/overcrowding fields created or assumed (Rule 4).
  - Missing values preserved as `NaN` and documented rather than filled with `0` (Rule 5).
  - All IDs preserved as strings to prevent truncation (Rule 6).
  - Clear, beginner-friendly cells and documentation throughout (Rule 7).

Ready for Module 3 (Route Inefficiency Scoring & Optimization).